# 01 · 전체 데이터 감사와 Manifest

이 파일은 DCC 2026 작업 노트북의 **공개용 보존본**입니다. 아래 원래 셀과 텍스트 출력은 저장된 실험 기록이며, 이번 공개 작업에서 모델 학습·추론·평가를 다시 실행하지 않았습니다.

- **목적:** Training ZIP 22쌍을 전수 검사하고 원본을 보존한 채 Manifest·격리 목록·감사 기록을 만듭니다.
- **입력:** 원천 ZIP 22개와 라벨 ZIP 22개, GCS 접근 권한
- **출력:** raw Manifest, archive/image/bbox 감사표, quarantine와 최종 seal(모든 gate 통과 시)
- **실제 저장된 실행 상태:** 22개 commit은 기록되어 있으나 C_0510의 image_format_failure_count=153, gate_pass=False로 최종 gate가 실패했습니다. 원본 01만으로 02의 승인된 v2 seal을 만들었다고 볼 수 없습니다.

**실행 전:** 공개 저장소에는 원천 이미지·라벨 ZIP, private test 연결표·정답, 모델 가중치, 원래 GCS 산출물이 없습니다. `YOUR_GCP_PROJECT_ID`, `YOUR_GCS_BUCKET`과 필요한 데이터·권한을 직접 설정해야 하며, 원래 봉인 파일의 generation·SHA-256·config 검증도 충족해야 합니다. placeholder만 바꾼다고 과거 산출물이 재생성되거나 모든 gate가 통과하는 것은 아닙니다.

**공개본 변경:** 개인 프로젝트·버킷·계정·로컬 데이터 경로를 placeholder로 바꾸고 Colab/Drive/작성자/widget metadata를 제거했습니다. 코드 셀의 순서와 계산·무결성 검증 로직은 보존했습니다. 모든 내장 image/HTML/JavaScript/기타 binary 출력과 markdown 첨부 이미지를 제거하고, 숫자·표·오류를 포함한 텍스트 출력은 식별자를 가린 뒤 보존했습니다. 사진이 없는 것은 해당 셀을 실행하지 않았다는 뜻이 아닙니다. 원본 code cell의 `execution_count`는 모두 `null`이므로 이 값만으로 실행 여부를 판단하지 않습니다.

[GitHub 코드](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/01_data_audit.ipynb) · [Colab에서 열기](https://colab.research.google.com/github/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/01_data_audit.ipynb) · [실행 관계와 상태](README.md) · [원본 이름·SHA-256 대응표](../docs/notebook-source-map.csv)

---

아래는 비공개 식별자를 치환한 원래 노트북입니다. 원래 안내의 예정 단계·성공 기준은 위의 실제 저장 상태와 구분해 읽으십시오.


# DCC 2026 고등부 예선 - 전체 Training 데이터 감사 및 Manifest v1

이 노트북은 GCS 업로드 감사를 통과한 Training ZIP 22쌍을 전수 검사하고,
재현 가능한 308,085행 raw Manifest와 품질 감사 산출물을 만듭니다.

## 입력 범위

- source: gs://버킷/dcc/raw/source/TS_*.zip 22개
- labels: gs://버킷/dcc/raw/labels/TL_*.zip 22개
- 공식 Validation의 VS_*.zip, VL_*.zip은 사용하지 않습니다.
- 원본 ZIP, 이미지, JSON에는 쓰기·수정·삭제를 수행하지 않습니다.

## 처리 방식

- CPU 런타임을 권장합니다.
- TS/TL 한 쌍만 /content에 내려받습니다.
- ZIP 전체를 loose file로 풀지 않고 member를 하나씩 읽습니다.
- GCS CRC32C, archive SHA-256, ZIP member CRC, 경로 안전성, stem 대응,
  JSON schema, strict image decode, bbox 품질을 검사합니다.
- key별 산출물을 검증한 뒤 create-only commit marker를 마지막에 기록합니다.
- 런타임이 끊기면 같은 RUN_ID로 다시 실행합니다. 검증된 완료 key는 건너뜁니다.

## 최종 성공 기준

- 완료 key 22개
- raw rows = unique stems = 308,085
- only-image = only-JSON = JSON parse/schema failure = 0
- 모든 strict decode 실패가 trainable=False이고 quarantine과 정확히 대응
- 입력 GCS 객체 generation·size·CRC32C가 실행 전후 동일
- 최종 Parquet 4개, quarantine JSON, manifest seal을 GCS에 게시

전체 작업은 여러 Colab 세션에 걸릴 수 있습니다. 같은 RUN_ID를 유지하고
한 시점에 하나의 Colab 세션만 이 run을 실행하십시오.


In [ ]:
# 사용자가 확인할 설정값입니다. 첫 실행 후 RUN_ID와 OUTPUT_VERSION을 유지해야 재개됩니다.
PROJECT_ID = "YOUR_GCP_PROJECT_ID"
BUCKET_NAME = "YOUR_GCS_BUCKET"

RUN_ID = "20260825-manifest-v1"
OUTPUT_VERSION = "v1"

# 빈 list는 완료되지 않은 모든 key를 순서대로 처리합니다.
KEYS_TO_RUN = []
MAX_KEYS_THIS_SESSION = None

# 기존 런타임이 확실히 종료됐을 때만 True로 바꾸십시오.
TAKE_OVER_ACTIVE_LEASE = False
LEASE_TTL_MINUTES = 360

# record loop가 모든 member를 끝까지 읽으며 CRC를 검증합니다.
# True는 별도 사전 진단으로 전체 ZIP을 한 번 더 읽습니다.
RUN_FULL_TESTZIP = False
STRICT_REGRESSION_CHECKS = True
CLEAN_LOCAL_AFTER_SUCCESS = True
CLEAN_LOCAL_AFTER_FAILURE = True
PROGRESS_EVERY = 1000


## 1. 환경·인증

위 설정을 확인한 뒤 아래 셀을 실행합니다. Google 인증 창에서는 이 프로젝트와
버킷을 만든 계정을 선택하십시오. 서비스 계정 키와 인증 토큰은 노트북에 기록하지 않습니다.
패키지 교체 뒤 런타임 다시 시작 안내가 나오면 Colab 메뉴에서 한 번 재시작하고
첫 셀부터 다시 실행하십시오.


In [ ]:
import importlib
import importlib.util
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as dist_version

if not (3, 10) <= sys.version_info[:2] <= (3, 14):
    raise RuntimeError(
        f"지원 Python은 3.10~3.14입니다: {sys.version_info.major}.{sys.version_info.minor}"
    )

REQUIRED_PACKAGES = {
    "google.cloud.storage": ("google-cloud-storage", "3.13.1"),
    "google_crc32c": ("google-crc32c", "1.8.0"),
    "pyarrow": ("pyarrow", "23.0.1"),
    "PIL": ("Pillow", "12.3.0"),
}
def module_available(module):
    try:
        return importlib.util.find_spec(module) is not None
    except (ImportError, ModuleNotFoundError):
        return False

def requirement_satisfied(distribution, expected_version):
    try:
        installed = dist_version(distribution)
    except PackageNotFoundError:
        return False
    return installed == expected_version

needs_install = [
    (module, f"{distribution}=={expected_version}")
    for module, (distribution, expected_version) in REQUIRED_PACKAGES.items()
    if (
        not module_available(module)
        or not requirement_satisfied(distribution, expected_version)
    )
]
missing = [requirement for _, requirement in needs_install]
preloaded_needing_replacement = [
    module for module, _ in needs_install if module in sys.modules
]
if missing:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *missing],
        check=True,
    )
    importlib.invalidate_caches()
    if preloaded_needing_replacement:
        raise RuntimeError(
            "교체한 패키지의 구버전 모듈이 메모리에 남아 있습니다: "
            f"{preloaded_needing_replacement}. Colab의 런타임 다시 시작을 누른 뒤 "
            "첫 셀부터 다시 실행하세요."
        )
unresolved = [
    f"{distribution}=={expected_version}"
    for distribution, expected_version in REQUIRED_PACKAGES.values()
    if not requirement_satisfied(distribution, expected_version)
]
if unresolved:
    raise RuntimeError(f"패키지 버전 요구사항을 충족하지 못했습니다: {unresolved}")

import base64
import csv
import gc
import hashlib
import io
import json
import math
import os
import platform
import re
import shutil
import stat
import tempfile
import time
import traceback
import uuid
import warnings
import zipfile
from collections import Counter
from datetime import datetime, timedelta, timezone
from importlib.metadata import version as package_version
from pathlib import Path

import google_crc32c
import PIL
import pyarrow as pa
import pyarrow.parquet as pq
from google.api_core.exceptions import NotFound, PreconditionFailed
from google.cloud import storage
from PIL import Image, ImageFile

from google.colab import auth

ImageFile.LOAD_TRUNCATED_IMAGES = False
# 정상 입력은 1920x1080입니다. 더 큰 header는 pixel allocation 전에 차단합니다.
Image.MAX_IMAGE_PIXELS = 1920 * 1080
assert ImageFile.LOAD_TRUNCATED_IMAGES is False
IMPORTED_PACKAGE_VERSIONS = {
    "google-cloud-storage": storage.__version__,
    "google-crc32c": getattr(
        google_crc32c, "__version__", package_version("google-crc32c")
    ),
    "pyarrow": pa.__version__,
    "Pillow": PIL.__version__,
}
expected_imported_versions = {
    distribution: expected_version
    for distribution, expected_version in REQUIRED_PACKAGES.values()
}
if IMPORTED_PACKAGE_VERSIONS != expected_imported_versions:
    raise RuntimeError(
        "import된 패키지 버전이 lock과 다릅니다. 런타임을 다시 시작하세요: "
        f"actual={IMPORTED_PACKAGE_VERSIONS}, expected={expected_imported_versions}"
    )
RUNTIME_ENVIRONMENT = {
    "python": f"{sys.version_info.major}.{sys.version_info.minor}",
    "pyarrow": IMPORTED_PACKAGE_VERSIONS["pyarrow"],
    "pillow": IMPORTED_PACKAGE_VERSIONS["Pillow"],
    "google_cloud_storage": IMPORTED_PACKAGE_VERSIONS["google-cloud-storage"],
    "google_crc32c": IMPORTED_PACKAGE_VERSIONS["google-crc32c"],
}

if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{2,63}", RUN_ID):
    raise ValueError("RUN_ID는 영문·숫자·점·밑줄·하이픈 3~64자로 지정하세요.")
if not re.fullmatch(r"v[0-9]+", OUTPUT_VERSION):
    raise ValueError("OUTPUT_VERSION은 v1 같은 형식으로 지정하세요.")
if BUCKET_NAME.startswith("gs://"):
    raise ValueError("BUCKET_NAME에는 gs://를 넣지 마세요.")

auth.authenticate_user(project_id=PROJECT_ID)
STORAGE_CLIENT = storage.Client(project=PROJECT_ID)
BUCKET = STORAGE_CLIENT.bucket(BUCKET_NAME)
BUCKET.reload()
if str(BUCKET.location or "").upper() != "US-CENTRAL1":
    raise RuntimeError(f"예상하지 않은 bucket location: {BUCKET.location}")
if str(BUCKET.storage_class or "").upper() != "STANDARD":
    raise RuntimeError(f"예상하지 않은 storage class: {BUCKET.storage_class}")
if not BUCKET.iam_configuration.uniform_bucket_level_access_enabled:
    raise RuntimeError("Uniform bucket-level access가 활성화되어 있지 않습니다.")
public_access_prevention = getattr(
    BUCKET.iam_configuration, "public_access_prevention", None
)
if str(public_access_prevention or "").lower() != "enforced":
    raise RuntimeError(
        f"Public access prevention이 enforced가 아닙니다: "
        f"{public_access_prevention}"
    )

print("Python:", RUNTIME_ENVIRONMENT["python"])
print("Platform:", platform.platform())
print("PyArrow:", RUNTIME_ENVIRONMENT["pyarrow"])
print("Pillow:", RUNTIME_ENVIRONMENT["pillow"])
print("google-cloud-storage:", RUNTIME_ENVIRONMENT["google_cloud_storage"])
disk = shutil.disk_usage("/content")
print(f"Local disk free: {disk.free / 1024**3:.1f} GiB")
print("Bucket:", f"gs://{BUCKET_NAME}")
print(
    "Bucket policy:",
    BUCKET.location,
    BUCKET.storage_class,
    "uniform-access",
    public_access_prevention,
)


Python: 3.13
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
PyArrow: 23.0.1
Pillow: 12.3.0
google-cloud-storage: 3.13.1
Local disk free: 188.2 GiB
Bucket: gs://YOUR_GCS_BUCKET
Bucket policy: US-CENTRAL1 STANDARD uniform-access enforced


In [ ]:
NOTEBOOK_NAME = "DCC_Colab_01_전체_데이터_감사_Manifest.ipynb"
PIPELINE_VERSION = "dcc-full-manifest-audit-1.1.0"
PIPELINE_BUILD_SHA256 = "dd582fa05398cf2b62e3b6249082263e02d862d089656c68ef85f8f77a3950f0"
SCHEMA_VERSION = "dcc-manifest-schema-v1"

RAW_SOURCE_PREFIX = "dcc/raw/source"
RAW_LABEL_PREFIX = "dcc/raw/labels"
RUN_PREFIX = f"dcc/runs/{RUN_ID}"
LOCAL_RUN_ROOT = Path("/content/dcc_manifest_runs") / RUN_ID
LOCAL_RUN_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_BY_KEY = {
    "A_0005": (6_691_689_410, 21_903_965, 24_068),
    "A_0510": (14_906_742_900, 42_457_491, 49_848),
    "A_1015": (11_165_900_539, 28_195_339, 31_304),
    "B_0003": (7_715_251_257, 17_171_560, 16_653),
    "B_0306": (5_661_978_328, 11_781_249, 13_811),
    "B_0609": (2_589_522_515, 5_780_771, 6_802),
    "C_0005": (9_087_926_484, 21_286_360, 22_335),
    "C_0510": (19_338_857_630, 33_127_640, 37_466),
    "C_1015": (15_482_673_854, 27_055_549, 32_140),
    "E_0003": (5_019_586_536, 7_700_792, 9_324),
    "E_0306": (2_762_769_824, 8_507_548, 7_008),
    "E_0610": (2_753_727_777, 6_480_125, 7_174),
    "F_0002": (1_567_936_700, 2_819_544, 3_404),
    "F_0204": (1_862_771_404, 4_546_233, 5_383),
    "G_0002": (2_265_269_014, 4_281_182, 4_666),
    "G_0204": (3_137_701_657, 5_206_889, 6_311),
    "H_0007": (766_466_626, 2_339_421, 2_839),
    "H_0714": (1_442_602_803, 3_325_010, 4_208),
    "H_1420": (1_278_235_340, 2_770_114, 3_264),
    "S_0007": (2_055_352_703, 4_054_375, 4_675),
    "S_0714": (3_803_208_130, 5_622_282, 6_378),
    "S_1420": (4_792_296_782, 7_472_664, 9_024),
}
EXPECTED_KEYS = tuple(EXPECTED_BY_KEY)
EXPECTED_TYPES = set("ABCEFGHS")
EXPECTED_SOURCE_BYTES = 126_148_468_213
EXPECTED_LABEL_BYTES = 273_886_103
EXPECTED_RAW_ROWS = 308_085
EXPECTED_ANNOTATIONS = 3_418_229
EXPECTED_SUFFIX_COUNTS = {".jpg": 298_587, ".jpeg": 9_498}
EXPECTED_MOKJA_COUNT_DISTRIBUTION = {0: 12, 1: 302_275, 2: 5_667, 3: 131}
EXPECTED_SURFACE_COUNT_DISTRIBUTION = {0: 25, 1: 306_942, 2: 1_077, 3: 41}
EXPECTED_BBOX_OUT_OF_RANGE = 631
EXPECTED_BBOX_OUT_OF_RANGE_FILES = 626
EXPECTED_BBOX_OUT_OF_RANGE_BY_NAME = {"mokja": 195, "surface": 434, "50cm": 2}
EXPECTED_OBSTRUCTION_TAGS = 407_452
EXPECTED_MULTI_OBSTRUCTION_ROWS = 76_538
EXPECTED_EMPTY_OBSTRUCTION_TAGS = 10
EXPECTED_OBSTRUCTION_CONTRADICTIONS = 574
EXPECTED_WATERLEVEL_MIN = 0.0
EXPECTED_WATERLEVEL_MAX = 19.8
EXPECTED_WATERLEVEL_UNIQUE = 166

KNOWN_H_0007 = {
    "image_name": "H_0007_6aa9acd6-6fad-4e74-8ad7-a6b2ae29cefe.jpg",
    "stem": "H_0007_6aa9acd6-6fad-4e74-8ad7-a6b2ae29cefe",
    "image_bytes": 243_073,
    "image_sha256": "0268adb1b3c13ffe0fd661cfb833e6fb13b415ae779e8ff3fab50725b71a3a7d",
    "source_zip_sha256": "d1de7f2c3ca4cecf07c45414dcc6ccb1013f405bbed5151c3cf3953203409e3f",
    "label_zip_sha256": "840fcb27f15d5d6446aeda867d68b30be460b3157e05399cad1f3e161f400ae9",
}

assert len(EXPECTED_BY_KEY) == 22
assert sum(v[0] for v in EXPECTED_BY_KEY.values()) == EXPECTED_SOURCE_BYTES
assert sum(v[1] for v in EXPECTED_BY_KEY.values()) == EXPECTED_LABEL_BYTES
assert sum(v[2] for v in EXPECTED_BY_KEY.values()) == EXPECTED_RAW_ROWS

MAX_IMAGE_MEMBER_BYTES = 64 * 1024**2
MAX_JSON_MEMBER_BYTES = 8 * 1024**2
MAX_COMPRESSION_RATIO = 1000.0
SOURCE_UNCOMPRESSED_FACTOR_LIMIT = 1.5
LABEL_UNCOMPRESSED_FACTOR_LIMIT = 25.0
EXPECTED_IMAGE_SIZE = (1920, 1080)
IMAGE_SUFFIXES = {".jpg", ".jpeg"}
LABEL_SUFFIXES = {".json"}

MANIFEST_SCHEMA = pa.schema([
    ("archive_key", pa.string()),
    ("stem", pa.string()),
    ("type", pa.string()),
    ("clslevel", pa.string()),
    ("waterlevel_m", pa.float64()),
    ("source_zip_uri", pa.string()),
    ("source_generation", pa.int64()),
    ("source_member", pa.string()),
    ("source_member_crc32", pa.string()),
    ("source_member_bytes", pa.int64()),
    ("label_zip_uri", pa.string()),
    ("label_generation", pa.int64()),
    ("label_member", pa.string()),
    ("label_member_crc32", pa.string()),
    ("label_member_bytes", pa.int64()),
    ("label_sha256", pa.string()),
    ("image_name", pa.string()),
    ("image_suffix", pa.string()),
    ("image_format", pa.string()),
    ("width", pa.int32()),
    ("height", pa.int32()),
    ("image_sha256", pa.string()),
    ("image_uuid", pa.string()),
    ("image_id", pa.string()),
    ("frame_id", pa.string()),
    ("json_parse_ok", pa.bool_()),
    ("schema_ok", pa.bool_()),
    ("strict_decode_ok", pa.bool_()),
    ("trainable", pa.bool_()),
    ("detector_trainable", pa.bool_()),
    ("surface_aux_trainable", pa.bool_()),
    ("quarantine_reason", pa.string()),
    ("obstruction", pa.list_(pa.string())),
    ("annotation_count", pa.int32()),
    ("mokja_count", pa.int32()),
    ("surface_count", pa.int32()),
    ("valid_mokja_count", pa.int32()),
    ("valid_surface_count", pa.int32()),
    ("bbox_out_of_range", pa.bool_()),
    ("bbox_degenerate", pa.bool_()),
    ("bbox_sanitized", pa.bool_()),
    ("bbox_issue_count", pa.int32()),
])

IMAGE_INTEGRITY_SCHEMA = pa.schema([
    ("archive_key", pa.string()),
    ("stem", pa.string()),
    ("source_zip_uri", pa.string()),
    ("source_member", pa.string()),
    ("image_name", pa.string()),
    ("image_suffix", pa.string()),
    ("source_member_bytes", pa.int64()),
    ("source_member_crc32", pa.string()),
    ("image_sha256", pa.string()),
    ("image_format", pa.string()),
    ("image_mode", pa.string()),
    ("width", pa.int32()),
    ("height", pa.int32()),
    ("strict_decode_ok", pa.bool_()),
    ("strict_error_type", pa.string()),
    ("strict_error_message", pa.string()),
    ("jpeg_soi_ok", pa.bool_()),
    ("jpeg_eoi_ok", pa.bool_()),
    ("format_matches_suffix", pa.bool_()),
    ("expected_resolution_ok", pa.bool_()),
    ("declared_width", pa.int32()),
    ("declared_height", pa.int32()),
    ("declared_size_matches", pa.bool_()),
    ("trainable", pa.bool_()),
    ("quarantine_reason", pa.string()),
])

BBOX_SCHEMA = pa.schema([
    ("archive_key", pa.string()),
    ("stem", pa.string()),
    ("annotation_index", pa.int32()),
    ("annotation_name", pa.string()),
    ("class_id", pa.int32()),
    ("parse_ok", pa.bool_()),
    ("finite_ok", pa.bool_()),
    ("center_x", pa.float64()),
    ("center_y", pa.float64()),
    ("width", pa.float64()),
    ("height", pa.float64()),
    ("x1", pa.float64()),
    ("y1", pa.float64()),
    ("x2", pa.float64()),
    ("y2", pa.float64()),
    ("clipped_x1", pa.float64()),
    ("clipped_y1", pa.float64()),
    ("clipped_x2", pa.float64()),
    ("clipped_y2", pa.float64()),
    ("out_of_range", pa.bool_()),
    ("crosses_frame", pa.bool_()),
    ("degenerate", pa.bool_()),
    ("sanitized", pa.bool_()),
    ("usable_after_clip", pa.bool_()),
    ("issue", pa.string()),
])

ARCHIVE_AUDIT_SCHEMA = pa.schema([
    ("run_id", pa.string()),
    ("archive_key", pa.string()),
    ("status", pa.string()),
    ("gate_pass", pa.bool_()),
    ("started_at_utc", pa.string()),
    ("finished_at_utc", pa.string()),
    ("duration_seconds", pa.float64()),
    ("source_zip_uri", pa.string()),
    ("label_zip_uri", pa.string()),
    ("source_expected_bytes", pa.int64()),
    ("source_actual_bytes", pa.int64()),
    ("label_expected_bytes", pa.int64()),
    ("label_actual_bytes", pa.int64()),
    ("source_generation", pa.int64()),
    ("label_generation", pa.int64()),
    ("source_gcs_crc32c", pa.string()),
    ("label_gcs_crc32c", pa.string()),
    ("source_zip_sha256", pa.string()),
    ("label_zip_sha256", pa.string()),
    ("source_member_count", pa.int64()),
    ("label_member_count", pa.int64()),
    ("pair_count", pa.int64()),
    ("only_image_count", pa.int64()),
    ("only_json_count", pa.int64()),
    ("duplicate_image_stem_count", pa.int64()),
    ("duplicate_json_stem_count", pa.int64()),
    ("zip_security_ok", pa.bool_()),
    ("source_member_crc_verified", pa.bool_()),
    ("label_member_crc_verified", pa.bool_()),
    ("json_parse_failure_count", pa.int64()),
    ("json_schema_failure_count", pa.int64()),
    ("strict_decode_failure_count", pa.int64()),
    ("unexpected_resolution_count", pa.int64()),
    ("image_format_failure_count", pa.int64()),
    ("quarantine_count", pa.int64()),
    ("annotation_count", pa.int64()),
    ("bbox_parse_error_count", pa.int64()),
    ("bbox_out_of_range_count", pa.int64()),
    ("bbox_crosses_frame_count", pa.int64()),
    ("bbox_degenerate_count", pa.int64()),
    ("bbox_sanitized_count", pa.int64()),
    ("original_inputs_unchanged", pa.bool_()),
])

def canonical_json_bytes(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")

SCHEMA_SIGNATURES = {
    "manifest": str(MANIFEST_SCHEMA),
    "image_integrity": str(IMAGE_INTEGRITY_SCHEMA),
    "bbox_quality": str(BBOX_SCHEMA),
    "archive_audit": str(ARCHIVE_AUDIT_SCHEMA),
}
PIPELINE_CONTRACT = {
    "pipeline_version": PIPELINE_VERSION,
    "pipeline_build_sha256": PIPELINE_BUILD_SHA256,
    "schema_version": SCHEMA_VERSION,
    "output_version": OUTPUT_VERSION,
    "runtime_environment": RUNTIME_ENVIRONMENT,
    "expected_by_key": EXPECTED_BY_KEY,
    "regression_expectations": {
        "raw_rows": EXPECTED_RAW_ROWS,
        "annotations": EXPECTED_ANNOTATIONS,
        "suffix_counts": EXPECTED_SUFFIX_COUNTS,
        "mokja_distribution": EXPECTED_MOKJA_COUNT_DISTRIBUTION,
        "surface_distribution": EXPECTED_SURFACE_COUNT_DISTRIBUTION,
        "bbox_out_of_range": EXPECTED_BBOX_OUT_OF_RANGE,
        "bbox_out_of_range_files": EXPECTED_BBOX_OUT_OF_RANGE_FILES,
        "bbox_out_of_range_by_name": EXPECTED_BBOX_OUT_OF_RANGE_BY_NAME,
        "obstruction_tags": EXPECTED_OBSTRUCTION_TAGS,
        "multi_obstruction_rows": EXPECTED_MULTI_OBSTRUCTION_ROWS,
        "empty_obstruction_tags": EXPECTED_EMPTY_OBSTRUCTION_TAGS,
        "obstruction_contradictions": EXPECTED_OBSTRUCTION_CONTRADICTIONS,
        "waterlevel_min": EXPECTED_WATERLEVEL_MIN,
        "waterlevel_max": EXPECTED_WATERLEVEL_MAX,
        "waterlevel_unique": EXPECTED_WATERLEVEL_UNIQUE,
    },
    "schemas": SCHEMA_SIGNATURES,
    "run_full_testzip": RUN_FULL_TESTZIP,
    "strict_regression_checks": STRICT_REGRESSION_CHECKS,
    "image_member_limit": MAX_IMAGE_MEMBER_BYTES,
    "json_member_limit": MAX_JSON_MEMBER_BYTES,
    "compression_ratio_limit": MAX_COMPRESSION_RATIO,
    "source_uncompressed_factor_limit": SOURCE_UNCOMPRESSED_FACTOR_LIMIT,
    "label_uncompressed_factor_limit": LABEL_UNCOMPRESSED_FACTOR_LIMIT,
    "expected_image_size": EXPECTED_IMAGE_SIZE,
    "official_validation_allowed": False,
    "raw_write_allowed": False,
}
CONFIG_SHA256 = hashlib.sha256(canonical_json_bytes(PIPELINE_CONTRACT)).hexdigest()
print("Pipeline contract SHA-256:", CONFIG_SHA256)


Pipeline contract SHA-256: e16292ba81dced873d4cdbe92e40541c1f93481b344f6329c71bd6779f16afe9


In [ ]:
SESSION_ID = uuid.uuid4().hex
LEASE_OBJECT = f"{RUN_PREFIX}/lease.json"
RUN_CONFIG_OBJECT = f"{RUN_PREFIX}/run_config.json"
CURRENT_LEASE_GENERATION = None

def utc_now():
    return datetime.now(timezone.utc)

def iso_utc(value=None):
    value = value or utc_now()
    return value.isoformat().replace("+00:00", "Z")

def gs_uri(object_name):
    return f"gs://{BUCKET_NAME}/{object_name}"

def blob_descriptor(blob):
    blob.reload()
    return {
        "object": blob.name,
        "uri": gs_uri(blob.name),
        "generation": int(blob.generation),
        "size": int(blob.size),
        "crc32c": blob.crc32c or "",
        "md5_hash": blob.md5_hash or "",
        "updated": iso_utc(blob.updated) if blob.updated else "",
    }

def read_json_blob(object_name):
    blob = BUCKET.blob(object_name)
    if not blob.exists():
        return None
    return json.loads(blob.download_as_bytes().decode("utf-8"))

def immutable_blob_descriptor(blob):
    descriptor = blob_descriptor(blob)
    metadata = blob.metadata or {}
    required = [
        "sha256", "crc32c_local", "run_id", "config_sha256", "artifact_type"
    ]
    missing = [field for field in required if not metadata.get(field)]
    if missing:
        raise IOError(
            f"불변 객체 lineage metadata 누락: {gs_uri(blob.name)} / {missing}"
        )
    row_count_text = metadata.get("row_count")
    descriptor.update({
        "sha256": metadata["sha256"],
        "crc32c_local": metadata["crc32c_local"],
        "run_id": metadata["run_id"],
        "config_sha256": metadata["config_sha256"],
        "artifact_type": metadata["artifact_type"],
        "row_count": (
            None if row_count_text is None else int(row_count_text)
        ),
    })
    return descriptor

def read_and_verify_immutable_json(object_name, artifact_type):
    blob = BUCKET.blob(object_name)
    blob.reload()
    generation = int(blob.generation)
    raw = blob.download_as_bytes(if_generation_match=generation)
    crc = google_crc32c.Checksum()
    crc.update(raw)
    raw_crc32c = base64.b64encode(crc.digest()).decode("ascii")
    raw_sha256 = hashlib.sha256(raw).hexdigest()
    metadata = blob.metadata or {}
    checks = {
        "size": int(blob.size) == len(raw),
        "gcs_crc32c": (blob.crc32c or "") == raw_crc32c,
        "sha256_metadata": metadata.get("sha256") == raw_sha256,
        "crc32c_local_metadata": metadata.get("crc32c_local") == raw_crc32c,
        "run_id_metadata": metadata.get("run_id") == RUN_ID,
        "config_sha256_metadata": metadata.get("config_sha256") == CONFIG_SHA256,
        "artifact_type_metadata": metadata.get("artifact_type") == artifact_type,
    }
    if not all(checks.values()):
        raise IOError(
            f"불변 JSON 자체 검증 실패: {gs_uri(object_name)} / {checks}"
        )
    descriptor = immutable_blob_descriptor(blob)
    if descriptor["generation"] != generation:
        raise IOError(f"불변 JSON 검증 중 generation 변경: {gs_uri(object_name)}")
    return json.loads(raw.decode("utf-8")), descriptor

def local_file_hashes(path, chunk_size=16 * 1024**2):
    path = Path(path)
    sha = hashlib.sha256()
    crc = google_crc32c.Checksum()
    with path.open("rb") as reader:
        for block in iter(lambda: reader.read(chunk_size), b""):
            sha.update(block)
            crc.update(block)
    return {
        "size": path.stat().st_size,
        "sha256": sha.hexdigest(),
        "crc32c": base64.b64encode(crc.digest()).decode("ascii"),
    }

def write_json_file(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True),
        encoding="utf-8",
    )

def upload_immutable(local_path, object_name, artifact_type, row_count=None):
    if object_name.startswith("dcc/raw/"):
        raise PermissionError("raw prefix에는 어떤 산출물도 쓸 수 없습니다.")
    local_path = Path(local_path)
    hashes = local_file_hashes(local_path)
    metadata = {
        "sha256": hashes["sha256"],
        "crc32c_local": hashes["crc32c"],
        "run_id": RUN_ID,
        "config_sha256": CONFIG_SHA256,
        "artifact_type": artifact_type,
    }
    if row_count is not None:
        metadata["row_count"] = str(int(row_count))

    blob = BUCKET.blob(object_name)
    blob.metadata = metadata
    try:
        blob.upload_from_filename(
            str(local_path),
            if_generation_match=0,
            checksum="crc32c",
        )
    except PreconditionFailed:
        blob.reload()
    blob.reload()

    remote_metadata = blob.metadata or {}
    if int(blob.size) != hashes["size"]:
        raise IOError(f"업로드 크기 불일치: {gs_uri(object_name)}")
    if blob.crc32c and blob.crc32c != hashes["crc32c"]:
        raise IOError(f"업로드 CRC32C 불일치: {gs_uri(object_name)}")
    metadata_checks = {
        field: remote_metadata.get(field) == expected
        for field, expected in metadata.items()
    }
    if not all(metadata_checks.values()):
        raise IOError(
            f"기존 불변 객체의 내용·계보 metadata 불일치: "
            f"{gs_uri(object_name)} / {metadata_checks}"
        )

    return {
        "object": object_name,
        "uri": gs_uri(object_name),
        "generation": int(blob.generation),
        "size": int(blob.size),
        "crc32c": blob.crc32c or "",
        "sha256": hashes["sha256"],
        "crc32c_local": hashes["crc32c"],
        "run_id": RUN_ID,
        "config_sha256": CONFIG_SHA256,
        "artifact_type": artifact_type,
        "row_count": None if row_count is None else int(row_count),
    }

def upload_json_create_only(payload, object_name, artifact_type):
    with tempfile.TemporaryDirectory() as temp_dir:
        local = Path(temp_dir) / "payload.json"
        write_json_file(local, payload)
        return upload_immutable(local, object_name, artifact_type)

def verify_remote_artifact(descriptor):
    blob = BUCKET.blob(descriptor["object"])
    blob.reload()
    metadata = blob.metadata or {}
    checks = {
        "generation": int(blob.generation) == int(descriptor["generation"]),
        "size": int(blob.size) == int(descriptor["size"]),
        "crc32c": (blob.crc32c or "") == descriptor["crc32c"],
        "crc32c_lineage": (
            descriptor["crc32c_local"] == descriptor["crc32c"]
        ),
        "sha256_metadata": metadata.get("sha256") == descriptor["sha256"],
        "crc32c_local_metadata": (
            metadata.get("crc32c_local") == descriptor["crc32c_local"]
        ),
        "run_id_metadata": metadata.get("run_id") == descriptor["run_id"],
        "config_sha256_metadata": (
            metadata.get("config_sha256") == descriptor["config_sha256"]
        ),
        "artifact_type_metadata": (
            metadata.get("artifact_type") == descriptor["artifact_type"]
        ),
    }
    if descriptor.get("row_count") is not None:
        checks["row_count_metadata"] = (
            metadata.get("row_count") == str(int(descriptor["row_count"]))
        )
    if not all(checks.values()):
        raise IOError(f"원격 산출물 검증 실패: {descriptor['uri']} / {checks}")
    return True

def collect_source_inventory():
    expected_names = set()
    inventory = []
    for key, (source_bytes, label_bytes, _) in EXPECTED_BY_KEY.items():
        source_name = f"{RAW_SOURCE_PREFIX}/TS_{key}.zip"
        label_name = f"{RAW_LABEL_PREFIX}/TL_{key}.zip"
        expected_names.update({source_name, label_name})
        for object_name, expected_size, kind in [
            (source_name, source_bytes, "source"),
            (label_name, label_bytes, "label"),
        ]:
            blob = BUCKET.blob(object_name)
            if not blob.exists():
                raise FileNotFoundError(gs_uri(object_name))
            desc = blob_descriptor(blob)
            if desc["size"] != expected_size:
                raise IOError(
                    f"{kind} exact byte 불일치: {desc['uri']} "
                    f"actual={desc['size']:,}, expected={expected_size:,}"
                )
            if not desc["crc32c"]:
                raise IOError(f"{kind} GCS CRC32C metadata 누락: {desc['uri']}")
            desc.update({"key": key, "kind": kind})
            inventory.append(desc)

    raw_objects = [blob.name for blob in STORAGE_CLIENT.list_blobs(BUCKET, prefix="dcc/raw/")]
    training_like = {
        name
        for name in raw_objects
        if Path(name).name.startswith(("TS_", "TL_"))
        and Path(name).suffix.lower() == ".zip"
    }
    validation_like = [
        name
        for name in raw_objects
        if Path(name).name.startswith(("VS_", "VL_"))
        and Path(name).suffix.lower() == ".zip"
    ]
    if training_like != expected_names:
        raise IOError(
            "raw Training 객체 집합 불일치: "
            f"missing={sorted(expected_names - training_like)}, "
            f"unexpected={sorted(training_like - expected_names)}"
        )
    if validation_like:
        print(
            "WARNING: 공식 Validation ZIP은 이번 Training audit에서 제외합니다:",
            validation_like[:10],
        )

    inventory.sort(key=lambda item: (item["key"], item["kind"]))
    inventory_sha = hashlib.sha256(canonical_json_bytes(inventory)).hexdigest()
    return inventory, inventory_sha

def ensure_run_config(inventory, inventory_sha):
    immutable = {
        "run_id": RUN_ID,
        "project_id": PROJECT_ID,
        "bucket_name": BUCKET_NAME,
        "run_prefix": RUN_PREFIX,
        "pipeline_contract": PIPELINE_CONTRACT,
        "config_sha256": CONFIG_SHA256,
        "source_inventory": inventory,
        "source_inventory_sha256": inventory_sha,
    }
    config_blob = BUCKET.blob(RUN_CONFIG_OBJECT)
    if config_blob.exists():
        existing = read_json_blob(RUN_CONFIG_OBJECT)
        for field in [
            "run_id",
            "project_id",
            "bucket_name",
            "config_sha256",
            "source_inventory_sha256",
        ]:
            if existing.get(field) != immutable.get(field):
                raise RuntimeError(
                    f"기존 RUN_ID의 {field}가 현재 설정과 다릅니다. "
                    "새 RUN_ID를 사용하십시오."
                )
        verified, _ = read_and_verify_immutable_json(
            RUN_CONFIG_OBJECT, "run_config"
        )
        return verified

    payload = {
        **immutable,
        "notebook_name": NOTEBOOK_NAME,
        "pipeline_version": PIPELINE_VERSION,
        "schema_version": SCHEMA_VERSION,
        "environment": {
            **RUNTIME_ENVIRONMENT,
        },
    }
    upload_json_create_only(payload, RUN_CONFIG_OBJECT, "run_config")
    verified, _ = read_and_verify_immutable_json(
        RUN_CONFIG_OBJECT, "run_config"
    )
    return verified

def _lease_payload(status="active"):
    now = utc_now()
    return {
        "run_id": RUN_ID,
        "session_id": SESSION_ID,
        "status": status,
        "updated_at_utc": iso_utc(now),
        "expires_at_utc": iso_utc(
            now + timedelta(minutes=LEASE_TTL_MINUTES)
            if status == "active"
            else now
        ),
    }

def acquire_run_lease():
    global CURRENT_LEASE_GENERATION
    blob = BUCKET.blob(LEASE_OBJECT)
    payload = _lease_payload("active")
    raw = canonical_json_bytes(payload)
    try:
        blob.upload_from_string(
            raw,
            content_type="application/json",
            if_generation_match=0,
            checksum="crc32c",
        )
    except PreconditionFailed:
        blob.reload()
        existing = json.loads(blob.download_as_bytes().decode("utf-8"))
        expires = datetime.fromisoformat(existing["expires_at_utc"].replace("Z", "+00:00"))
        active_other = (
            existing.get("status") == "active"
            and expires > utc_now()
            and existing.get("session_id") != SESSION_ID
        )
        if active_other and not TAKE_OVER_ACTIVE_LEASE:
            raise RuntimeError(
                "같은 RUN_ID를 다른 세션이 사용 중입니다. 기존 런타임을 종료하고 "
                "lease 만료 후 재시도하거나, 종료를 확인한 뒤 TAKE_OVER_ACTIVE_LEASE=True로 설정하세요."
            )
        generation = int(blob.generation)
        blob.upload_from_string(
            raw,
            content_type="application/json",
            if_generation_match=generation,
            checksum="crc32c",
        )
    blob.reload()
    CURRENT_LEASE_GENERATION = int(blob.generation)
    print("Run lease acquired:", gs_uri(LEASE_OBJECT), "session", SESSION_ID[:8])

def refresh_run_lease():
    global CURRENT_LEASE_GENERATION
    if CURRENT_LEASE_GENERATION is None:
        raise RuntimeError("run lease를 먼저 획득하세요.")
    blob = BUCKET.blob(LEASE_OBJECT)
    blob.upload_from_string(
        canonical_json_bytes(_lease_payload("active")),
        content_type="application/json",
        if_generation_match=CURRENT_LEASE_GENERATION,
        checksum="crc32c",
    )
    blob.reload()
    CURRENT_LEASE_GENERATION = int(blob.generation)

def release_run_lease():
    global CURRENT_LEASE_GENERATION
    if CURRENT_LEASE_GENERATION is None:
        return
    blob = BUCKET.blob(LEASE_OBJECT)
    try:
        blob.upload_from_string(
            canonical_json_bytes(_lease_payload("released")),
            content_type="application/json",
            if_generation_match=CURRENT_LEASE_GENERATION,
            checksum="crc32c",
        )
        blob.reload()
        print("Run lease released.")
    except PreconditionFailed:
        print("WARNING: lease generation이 바뀌어 release를 건너뜁니다.")
    except Exception as exc:
        print("WARNING: run lease 해제 요청 실패:", repr(exc))
    finally:
        CURRENT_LEASE_GENERATION = None


In [ ]:
class AuditError(RuntimeError):
    pass

def normalize_flat_member(raw_name):
    if not isinstance(raw_name, str):
        raise AuditError("ZIP member 이름이 문자열이 아닙니다.")
    if "\x00" in raw_name or "\\" in raw_name or raw_name.startswith("//"):
        raise AuditError(f"안전하지 않은 ZIP member: {raw_name!r}")
    name = raw_name[1:] if raw_name.startswith("/") else raw_name
    if (
        not name
        or "/" in name
        or name in {".", ".."}
        or ":" in name
        or Path(name).name != name
    ):
        raise AuditError(f"중첩 또는 안전하지 않은 ZIP member: {raw_name!r}")
    return name

def inspect_zip_layout(
    zip_path,
    allowed_suffixes,
    archive_key,
    max_member_bytes,
    expected_member_count=None,
    max_total_uncompressed_bytes=None,
):
    mapping = {}
    folded_names = set()
    folded_stems = set()
    with zipfile.ZipFile(zip_path) as zf:
        infos = zf.infolist()
        if not infos:
            raise AuditError(f"빈 ZIP: {zip_path}")
        if (
            expected_member_count is not None
            and len(infos) != int(expected_member_count)
        ):
            raise AuditError(
                f"ZIP member 수 불일치: {Path(zip_path).name} / "
                f"{len(infos):,} != {int(expected_member_count):,}"
            )
        total_uncompressed = sum(max(0, int(info.file_size)) for info in infos)
        if (
            max_total_uncompressed_bytes is not None
            and total_uncompressed > int(max_total_uncompressed_bytes)
        ):
            raise AuditError(
                f"ZIP 총 비압축 크기 제한 위반: {Path(zip_path).name} / "
                f"{total_uncompressed:,} > {int(max_total_uncompressed_bytes):,}"
            )
        for info in infos:
            if info.is_dir():
                raise AuditError(f"ZIP directory entry 금지: {info.filename!r}")
            if info.flag_bits & 0x1:
                raise AuditError(f"암호화 ZIP member 금지: {info.filename!r}")
            mode = (info.external_attr >> 16) & 0xFFFF
            if stat.S_IFMT(mode) not in {0, stat.S_IFREG}:
                raise AuditError(f"symlink 또는 special member 금지: {info.filename!r}")
            name = normalize_flat_member(info.filename)
            suffix = Path(name).suffix.lower()
            if suffix not in allowed_suffixes:
                raise AuditError(f"예상 밖 ZIP 확장자: {name}")
            if info.file_size < 0 or info.file_size > max_member_bytes:
                raise AuditError(
                    f"ZIP member 크기 제한 위반: {name} / {info.file_size:,} bytes"
                )
            ratio = info.file_size / max(1, info.compress_size)
            if ratio > MAX_COMPRESSION_RATIO:
                raise AuditError(f"비정상 압축률: {name} / {ratio:.1f}")
            folded_name = name.casefold()
            stem = Path(name).stem
            folded_stem = stem.casefold()
            if folded_name in folded_names:
                raise AuditError(f"casefold 중복 파일명: {name}")
            if folded_stem in folded_stems:
                raise AuditError(f"casefold 중복 stem: {stem}")
            if not stem.startswith(f"{archive_key}_"):
                raise AuditError(
                    f"archive key와 member prefix 불일치: {archive_key} / {name}"
                )
            folded_names.add(folded_name)
            folded_stems.add(folded_stem)
            mapping[stem] = {
                "name": name,
                "info": info,
                "member_crc32": f"{info.CRC:08x}",
                "member_bytes": int(info.file_size),
            }

        if RUN_FULL_TESTZIP:
            bad_member = zf.testzip()
            if bad_member is not None:
                raise zipfile.BadZipFile(
                    f"member CRC 오류: {Path(zip_path).name} / {bad_member}"
                )
    return mapping

def reject_json_constant(value):
    raise ValueError(f"허용되지 않는 JSON 상수: {value}")

def reject_duplicate_json_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"중복 JSON key: {key}")
        result[key] = value
    return result

def strict_json_loads(text):
    return json.loads(
        text,
        parse_constant=reject_json_constant,
        object_pairs_hook=reject_duplicate_json_keys,
    )

def finite_float(value, field_name):
    if isinstance(value, bool):
        raise ValueError(f"{field_name}에 bool을 사용할 수 없습니다.")
    number = float(value)
    if not math.isfinite(number):
        raise ValueError(f"{field_name}가 유한한 수가 아닙니다.")
    return number

def inspect_image_bytes(raw, suffix):
    image_sha256 = hashlib.sha256(raw).hexdigest()
    jpeg_soi_ok = raw.startswith(b"\xff\xd8")
    jpeg_eoi_ok = raw.endswith(b"\xff\xd9")
    result = {
        "image_sha256": image_sha256,
        "image_format": None,
        "image_mode": None,
        "width": None,
        "height": None,
        "strict_decode_ok": False,
        "strict_error_type": "",
        "strict_error_message": "",
        "jpeg_soi_ok": jpeg_soi_ok,
        "jpeg_eoi_ok": jpeg_eoi_ok,
        "format_matches_suffix": False,
        "expected_resolution_ok": False,
    }
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error")
            with Image.open(io.BytesIO(raw)) as image:
                result["image_format"] = image.format
                result["width"], result["height"] = image.size
                if image.size != EXPECTED_IMAGE_SIZE:
                    raise AuditError(
                        f"예상 밖 image header 크기: {image.size}"
                    )
                image.verify()
            with Image.open(io.BytesIO(raw)) as image:
                if image.size != EXPECTED_IMAGE_SIZE:
                    raise AuditError(
                        f"pixel load 전 image 크기 제한 위반: {image.size}"
                    )
                image.load()
                result["image_mode"] = image.mode
                result["width"], result["height"] = image.size
        result["strict_decode_ok"] = True
    except Exception as exc:
        result["strict_error_type"] = type(exc).__name__
        result["strict_error_message"] = str(exc)[:2000]
        if result["width"] is None:
            try:
                with Image.open(io.BytesIO(raw)) as image:
                    result["image_format"] = image.format
                    result["width"], result["height"] = image.size
            except Exception:
                pass

    result["format_matches_suffix"] = (
        suffix in {".jpg", ".jpeg"} and result["image_format"] == "JPEG"
    )
    result["expected_resolution_ok"] = (
        result["width"] == 1920 and result["height"] == 1080
    )
    return result

def parse_bbox(annotation, archive_key, stem, annotation_index):
    row = {
        "archive_key": archive_key,
        "stem": stem,
        "annotation_index": int(annotation_index),
        "annotation_name": "",
        "class_id": None,
        "parse_ok": False,
        "finite_ok": False,
        "center_x": None,
        "center_y": None,
        "width": None,
        "height": None,
        "x1": None,
        "y1": None,
        "x2": None,
        "y2": None,
        "clipped_x1": None,
        "clipped_y1": None,
        "clipped_x2": None,
        "clipped_y2": None,
        "out_of_range": False,
        "crosses_frame": False,
        "degenerate": True,
        "sanitized": False,
        "usable_after_clip": False,
        "issue": "",
    }
    issues = []
    try:
        if not isinstance(annotation, dict):
            raise TypeError("annotation root가 object가 아님")
        annotation_name = annotation.get("name")
        if not isinstance(annotation_name, str) or not annotation_name:
            raise TypeError("annotation name이 비어 있거나 문자열이 아님")
        row["annotation_name"] = annotation_name
        class_id = annotation.get("class_id")
        if class_id is not None:
            if type(class_id) is not int:
                raise TypeError("class_id가 정수가 아님")
            row["class_id"] = class_id
        coordinates = annotation.get("relative_coordinates")
        if not isinstance(coordinates, str):
            raise TypeError("relative_coordinates가 JSON 문자열이 아님")
        coordinates = strict_json_loads(coordinates)
        if not isinstance(coordinates, dict):
            raise TypeError("relative_coordinates가 object가 아님")
        cx = finite_float(coordinates["center_x"], "center_x")
        cy = finite_float(coordinates["center_y"], "center_y")
        width = finite_float(coordinates["width"], "width")
        height = finite_float(coordinates["height"], "height")
        row.update({
            "parse_ok": True,
            "finite_ok": True,
            "center_x": cx,
            "center_y": cy,
            "width": width,
            "height": height,
        })
        x1, y1 = cx - width / 2, cy - height / 2
        x2, y2 = cx + width / 2, cy + height / 2
        clipped_x1 = min(1.0, max(0.0, x1))
        clipped_y1 = min(1.0, max(0.0, y1))
        clipped_x2 = min(1.0, max(0.0, x2))
        clipped_y2 = min(1.0, max(0.0, y2))
        out_of_range = not (
            0 <= cx <= 1
            and 0 <= cy <= 1
            and 0 < width <= 1
            and 0 < height <= 1
        )
        crosses_frame = x1 < 0 or y1 < 0 or x2 > 1 or y2 > 1
        positive_raw = width > 0 and height > 0
        positive_clipped = (
            clipped_x2 > clipped_x1 and clipped_y2 > clipped_y1
        )
        degenerate = not positive_raw or not positive_clipped
        sanitized = crosses_frame and positive_raw and positive_clipped
        usable = positive_raw and positive_clipped
        if width <= 0 or height <= 0:
            issues.append("non_positive_size")
        if out_of_range:
            issues.append("parameter_out_of_range")
        if crosses_frame:
            issues.append("crosses_frame")
        if not positive_clipped:
            issues.append("empty_after_clip")
        row.update({
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "clipped_x1": clipped_x1,
            "clipped_y1": clipped_y1,
            "clipped_x2": clipped_x2,
            "clipped_y2": clipped_y2,
            "out_of_range": out_of_range,
            "crosses_frame": crosses_frame,
            "degenerate": degenerate,
            "sanitized": sanitized,
            "usable_after_clip": usable,
        })
    except Exception as exc:
        issues.append(f"parse_error:{type(exc).__name__}:{str(exc)[:500]}")
    row["issue"] = ";".join(issues)
    return row

def parse_label_bytes(raw, expected_stem, expected_image_name, archive_key):
    result = {
        "json_parse_ok": False,
        "schema_ok": False,
        "schema_errors": [],
        "image_filename": "",
        "declared_width": None,
        "declared_height": None,
        "waterlevel_m": None,
        "type": archive_key[0],
        "clslevel": "",
        "obstruction": [],
        "image_uuid": "",
        "image_id": "",
        "frame_id": "",
        "bbox_rows": [],
        "annotation_count": 0,
        "mokja_count": 0,
        "surface_count": 0,
        "valid_mokja_count": 0,
        "valid_surface_count": 0,
    }
    try:
        text = raw.decode("utf-8-sig", errors="strict")
        record = strict_json_loads(text)
        result["json_parse_ok"] = True
    except Exception as exc:
        result["schema_errors"].append(
            f"json_parse_failed:{type(exc).__name__}:{str(exc)[:1000]}"
        )
        return result

    if not isinstance(record, dict):
        result["schema_errors"].append("json_root_not_object")
        return result

    image_info = record.get("image")
    annotations = record.get("annotations")
    categories = record.get("categories")
    if not isinstance(image_info, dict):
        result["schema_errors"].append("image_not_object")
        image_info = {}
    if not isinstance(annotations, list):
        result["schema_errors"].append("annotations_not_list")
        annotations = []
    if not isinstance(categories, list) or len(categories) != 1:
        result["schema_errors"].append("categories_must_have_exactly_one_item")
        categories = []

    image_filename_value = image_info.get("filename", "")
    if not isinstance(image_filename_value, str):
        result["schema_errors"].append("image_filename_not_string")
        image_filename_value = ""
    image_filename = image_filename_value
    result["image_filename"] = image_filename
    if (
        not image_filename
        or Path(image_filename).name != image_filename
        or "/" in image_filename
        or "\\" in image_filename
    ):
        result["schema_errors"].append("image_filename_not_basename")
    elif image_filename != expected_image_name:
        result["schema_errors"].append("image_filename_member_mismatch")
    if Path(image_filename).stem != expected_stem:
        result["schema_errors"].append("image_filename_stem_mismatch")

    for field in ["width", "height"]:
        value = image_info.get(field)
        if type(value) is int and value > 0:
            result[f"declared_{field}"] = value
        else:
            result["schema_errors"].append(f"invalid_image_{field}")
    waterlevel_value = image_info.get("waterlevel")
    try:
        if not isinstance(waterlevel_value, str):
            raise TypeError("waterlevel이 문자열이 아님")
        result["waterlevel_m"] = finite_float(waterlevel_value, "waterlevel")
    except Exception:
        result["schema_errors"].append("invalid_waterlevel")

    obstruction = image_info.get("obstruction", [])
    if not isinstance(obstruction, list):
        result["schema_errors"].append("obstruction_not_list")
    else:
        invalid_obstruction = [
            index
            for index, value in enumerate(obstruction)
            if not isinstance(value, str)
        ]
        if invalid_obstruction:
            result["schema_errors"].append(
                f"obstruction_item_not_string:{invalid_obstruction[:20]}"
            )
        result["obstruction"] = [
            value for value in obstruction if isinstance(value, str)
        ]

    result["image_uuid"] = str(image_info.get("uuid", "") or "")
    result["image_id"] = str(image_info.get("id", "") or "")
    result["frame_id"] = str(image_info.get("frameid", "") or "")

    category = categories[0] if categories and isinstance(categories[0], dict) else {}
    if categories and not isinstance(categories[0], dict):
        result["schema_errors"].append("category_not_object")
    category_type_value = category.get("mokjaType", "")
    if not isinstance(category_type_value, str):
        result["schema_errors"].append("mokja_type_not_string")
        category_type_value = ""
    category_type = category_type_value
    result["type"] = category_type or archive_key[0]
    clslevel_value = category.get("clslevel", "")
    if not isinstance(clslevel_value, str):
        result["schema_errors"].append("clslevel_not_string")
        clslevel_value = ""
    result["clslevel"] = clslevel_value
    if category_type != archive_key[0]:
        result["schema_errors"].append("mokja_type_archive_mismatch")
    if category_type not in EXPECTED_TYPES:
        result["schema_errors"].append("unexpected_mokja_type")
    if not result["clslevel"]:
        result["schema_errors"].append("missing_clslevel")

    result["annotation_count"] = len(annotations)
    for index, annotation in enumerate(annotations):
        bbox_row = parse_bbox(annotation, archive_key, expected_stem, index)
        result["bbox_rows"].append(bbox_row)
        name = bbox_row["annotation_name"]
        if name == "mokja":
            result["mokja_count"] += 1
            if bbox_row["usable_after_clip"]:
                result["valid_mokja_count"] += 1
        elif name == "surface":
            result["surface_count"] += 1
            if bbox_row["usable_after_clip"]:
                result["valid_surface_count"] += 1

    result["schema_ok"] = not result["schema_errors"]
    return result

class BufferedParquetWriter:
    def __init__(self, path, schema, batch_rows=20_000):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.schema = schema
        self.batch_rows = batch_rows
        self.buffer = []
        self.writer = pq.ParquetWriter(
            self.path,
            schema,
            compression="zstd",
            use_dictionary=True,
            write_statistics=True,
        )
        self.row_count = 0
        self.closed = False

    def append(self, row):
        self.buffer.append(row)
        if len(self.buffer) >= self.batch_rows:
            self.flush()

    def flush(self):
        if not self.buffer:
            return
        table = pa.Table.from_pylist(self.buffer, schema=self.schema)
        self.writer.write_table(table)
        self.row_count += len(self.buffer)
        self.buffer.clear()

    def close(self):
        if self.closed:
            return
        try:
            try:
                self.flush()
            finally:
                self.writer.close()
        finally:
            self.closed = True

def write_single_row_parquet(path, schema, row):
    table = pa.Table.from_pylist([row], schema=schema)
    pq.write_table(
        table,
        path,
        compression="zstd",
        use_dictionary=True,
        write_statistics=True,
    )

def safe_remove_tree(path):
    path = Path(path).resolve()
    root = LOCAL_RUN_ROOT.resolve()
    if path == root or root not in path.parents:
        raise RuntimeError(f"삭제 허용 범위를 벗어난 경로: {path}")
    if path.exists():
        shutil.rmtree(path)

def download_generation_pinned(
    blob,
    destination,
    expected_size,
    expected_generation,
    expected_crc32c,
):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    part = destination.with_name(destination.name + ".part")
    if part.exists():
        part.unlink()
    blob.download_to_filename(
        str(part),
        if_generation_match=int(expected_generation),
        checksum="crc32c",
    )
    if part.stat().st_size != int(expected_size):
        raise IOError(
            f"다운로드 byte 불일치: {blob.name} / "
            f"{part.stat().st_size:,} != {int(expected_size):,}"
        )
    hashes = local_file_hashes(part)
    if hashes["crc32c"] != expected_crc32c:
        raise IOError(f"다운로드 CRC32C 불일치: {blob.name}")
    os.replace(part, destination)
    return {**hashes, "generation": int(expected_generation)}


In [ ]:
# GCS를 사용하지 않는 빠른 helper 회귀 테스트입니다.
assert normalize_flat_member("A_0005_x.jpg") == "A_0005_x.jpg"
assert normalize_flat_member("/A_0005_x.jpg") == "A_0005_x.jpg"
for bad in ["//A.jpg", "../A.jpg", "a/b.jpg", "a\\b.jpg", "C:A.jpg", "\x00A.jpg"]:
    try:
        normalize_flat_member(bad)
        raise AssertionError(f"거부되지 않은 위험 경로: {bad!r}")
    except AuditError:
        pass

normal = parse_bbox(
    {"name": "mokja", "relative_coordinates": json.dumps({
        "center_x": 0.5, "center_y": 0.5, "width": 0.2, "height": 0.4
    })},
    "A_0005", "A_0005_test", 0,
)
assert normal["parse_ok"] and normal["usable_after_clip"]
assert not normal["out_of_range"] and not normal["degenerate"]

crossing = parse_bbox(
    {"name": "surface", "relative_coordinates": json.dumps({
        "center_x": 0.95, "center_y": 0.5, "width": 0.2, "height": 0.4
    })},
    "A_0005", "A_0005_test", 1,
)
assert not crossing["out_of_range"] and crossing["crosses_frame"]
assert crossing["sanitized"]
assert crossing["usable_after_clip"] and not crossing["degenerate"]

outside = parse_bbox(
    {"name": "surface", "relative_coordinates": json.dumps({
        "center_x": 2.0, "center_y": 0.5, "width": 0.2, "height": 0.4
    })},
    "A_0005", "A_0005_test", 2,
)
assert outside["out_of_range"] and outside["crosses_frame"]
assert outside["degenerate"]
assert not outside["usable_after_clip"]

image_buffer = io.BytesIO()
Image.new("RGB", EXPECTED_IMAGE_SIZE, color=(10, 20, 30)).save(
    image_buffer, format="JPEG"
)
valid_image = inspect_image_bytes(image_buffer.getvalue(), ".jpg")
assert valid_image["strict_decode_ok"] and valid_image["image_format"] == "JPEG"
valid_jpeg_bytes = image_buffer.getvalue()
truncated_image = inspect_image_bytes(
    valid_jpeg_bytes[: len(valid_jpeg_bytes) // 2], ".jpg"
)
assert not truncated_image["strict_decode_ok"]

print("Pure helper regression tests: PASS")


Pure helper regression tests: PASS


In [ ]:
def input_descriptor_for_key(key, kind):
    expected_source, expected_label, _ = EXPECTED_BY_KEY[key]
    if kind == "source":
        object_name = f"{RAW_SOURCE_PREFIX}/TS_{key}.zip"
        expected = expected_source
    elif kind == "label":
        object_name = f"{RAW_LABEL_PREFIX}/TL_{key}.zip"
        expected = expected_label
    else:
        raise ValueError(kind)
    blob = BUCKET.blob(object_name)
    desc = blob_descriptor(blob)
    if desc["size"] != expected:
        raise IOError(f"입력 byte 불일치: {desc['uri']}")
    return desc

def commit_object_name(key):
    return f"{RUN_PREFIX}/commits/{key}.json"

def verify_commit(key):
    object_name = commit_object_name(key)
    if not BUCKET.blob(object_name).exists():
        return None
    commit, _ = read_and_verify_immutable_json(object_name, "key_commit")
    if commit.get("run_id") != RUN_ID or commit.get("key") != key:
        raise IOError(f"commit identity 불일치: {key}")
    if commit.get("config_sha256") != CONFIG_SHA256:
        raise IOError(f"commit config 불일치: {key}")
    if commit.get("source_inventory_sha256") != SOURCE_INVENTORY_SHA256:
        raise IOError(f"commit source inventory 불일치: {key}")
    current_source = input_descriptor_for_key(key, "source")
    current_label = input_descriptor_for_key(key, "label")
    for current, captured, kind in [
        (current_source, commit["inputs"]["source"], "source"),
        (current_label, commit["inputs"]["label"], "label"),
    ]:
        for field in ["object", "generation", "size", "crc32c"]:
            if current[field] != captured[field]:
                raise IOError(f"{key} {kind} 입력이 commit 이후 변경됨: {field}")
    for descriptor in commit["artifacts"].values():
        verify_remote_artifact(descriptor)
    return commit

def upload_failure_report(key, attempt_id, started_at, exc):
    payload = {
        "status": "FAILED",
        "run_id": RUN_ID,
        "session_id": SESSION_ID,
        "attempt_id": attempt_id,
        "key": key,
        "config_sha256": CONFIG_SHA256,
        "source_inventory_sha256": SOURCE_INVENTORY_SHA256,
        "started_at_utc": started_at,
        "failed_at_utc": iso_utc(),
        "error_type": type(exc).__name__,
        "error_message": str(exc),
        "traceback": traceback.format_exc(),
    }
    object_name = (
        f"{RUN_PREFIX}/failures/{key}/"
        f"{attempt_id}-{uuid.uuid4().hex[:8]}.json"
    )
    return upload_json_create_only(payload, object_name, "failure_report")

def process_archive_pair(key):
    existing = verify_commit(key)
    if existing is not None:
        print(f"SKIP verified committed key: {key}")
        return existing

    started_clock = time.perf_counter()
    started_at = iso_utc()
    attempt_id = (
        f"{utc_now().strftime('%Y%m%dT%H%M%SZ')}-"
        f"{SESSION_ID[:8]}-{uuid.uuid4().hex[:8]}"
    )
    attempt_prefix = f"{RUN_PREFIX}/attempts/{key}/{attempt_id}"
    stage = LOCAL_RUN_ROOT / key / attempt_id
    if stage.exists():
        safe_remove_tree(stage)
    stage.mkdir(parents=True, exist_ok=False)

    source_local = stage / f"TS_{key}.zip"
    label_local = stage / f"TL_{key}.zip"
    manifest_local = stage / "raw_manifest.parquet"
    image_integrity_local = stage / "image_integrity.parquet"
    bbox_local = stage / "bbox_quality.parquet"
    archive_audit_local = stage / "archive_audit.parquet"
    quarantine_local = stage / "quarantine.json"
    open_writers = []

    try:
        refresh_run_lease()
        source_input = input_descriptor_for_key(key, "source")
        label_input = input_descriptor_for_key(key, "label")
        expected_source, expected_label, expected_pairs = EXPECTED_BY_KEY[key]

        required = expected_source + expected_label + 5 * 1024**3
        free = shutil.disk_usage("/content").free
        if free < required:
            raise RuntimeError(
                f"{key} 처리 공간 부족: free={free/1024**3:.1f} GiB, "
                f"required={required/1024**3:.1f} GiB"
            )

        print(f"\n[{key}] download source ({expected_source/1024**3:.3f} GiB)")
        source_download = download_generation_pinned(
            BUCKET.blob(source_input["object"]),
            source_local,
            expected_source,
            source_input["generation"],
            source_input["crc32c"],
        )
        print(f"[{key}] download labels ({expected_label/1024**3:.3f} GiB)")
        label_download = download_generation_pinned(
            BUCKET.blob(label_input["object"]),
            label_local,
            expected_label,
            label_input["generation"],
            label_input["crc32c"],
        )
        refresh_run_lease()

        if key == "H_0007":
            assert source_download["sha256"] == KNOWN_H_0007["source_zip_sha256"]
            assert label_download["sha256"] == KNOWN_H_0007["label_zip_sha256"]

        print(
            f"[{key}] inspect ZIP directory/security; "
            "record read verifies every member CRC"
        )
        source_map = inspect_zip_layout(
            source_local,
            IMAGE_SUFFIXES,
            key,
            MAX_IMAGE_MEMBER_BYTES,
            expected_member_count=expected_pairs,
            max_total_uncompressed_bytes=(
                expected_source * SOURCE_UNCOMPRESSED_FACTOR_LIMIT
                + MAX_IMAGE_MEMBER_BYTES
            ),
        )
        label_map = inspect_zip_layout(
            label_local,
            LABEL_SUFFIXES,
            key,
            MAX_JSON_MEMBER_BYTES,
            expected_member_count=expected_pairs,
            max_total_uncompressed_bytes=(
                expected_label * LABEL_UNCOMPRESSED_FACTOR_LIMIT
                + MAX_JSON_MEMBER_BYTES
            ),
        )
        source_stems = set(source_map)
        label_stems = set(label_map)
        only_image = sorted(source_stems - label_stems)
        only_json = sorted(label_stems - source_stems)
        if only_image or only_json:
            raise AuditError(
                f"{key} stem pair 불일치: "
                f"only_image={only_image[:5]}, only_json={only_json[:5]}"
            )
        if len(source_map) != expected_pairs:
            raise AuditError(
                f"{key} pair count 불일치: {len(source_map):,} != {expected_pairs:,}"
            )
        refresh_run_lease()

        manifest_writer = BufferedParquetWriter(
            manifest_local, MANIFEST_SCHEMA, batch_rows=10_000
        )
        open_writers.append(manifest_writer)
        integrity_writer = BufferedParquetWriter(
            image_integrity_local, IMAGE_INTEGRITY_SCHEMA, batch_rows=10_000
        )
        open_writers.append(integrity_writer)
        bbox_writer = BufferedParquetWriter(
            bbox_local, BBOX_SCHEMA, batch_rows=20_000
        )
        open_writers.append(bbox_writer)

        counters = Counter()
        quarantine_items = []
        started_records = time.perf_counter()

        with zipfile.ZipFile(source_local) as source_zip, zipfile.ZipFile(label_local) as label_zip:
            for index, stem in enumerate(sorted(source_map), start=1):
                source_entry = source_map[stem]
                label_entry = label_map[stem]
                label_raw = label_zip.read(label_entry["info"])
                image_raw = source_zip.read(source_entry["info"])
                label_sha256 = hashlib.sha256(label_raw).hexdigest()
                suffix = Path(source_entry["name"]).suffix.lower()
                image_result = inspect_image_bytes(image_raw, suffix)
                label_result = parse_label_bytes(
                    label_raw,
                    expected_stem=stem,
                    expected_image_name=source_entry["name"],
                    archive_key=key,
                )

                for bbox_row in label_result["bbox_rows"]:
                    bbox_writer.append(bbox_row)
                    counters["annotation_count"] += 1
                    counters["bbox_parse_error_count"] += int(not bbox_row["parse_ok"])
                    counters["bbox_out_of_range_count"] += int(bbox_row["out_of_range"])
                    counters["bbox_crosses_frame_count"] += int(
                        bbox_row["crosses_frame"]
                    )
                    counters["bbox_degenerate_count"] += int(bbox_row["degenerate"])
                    counters["bbox_sanitized_count"] += int(bbox_row["sanitized"])

                reasons = []
                if not label_result["json_parse_ok"]:
                    reasons.append("json_parse_failed")
                if not label_result["schema_ok"]:
                    reasons.append("json_schema_failed")
                if not image_result["strict_decode_ok"]:
                    reasons.append("strict_decode_failed")
                if (
                    image_result["image_format"] is not None
                    and not image_result["format_matches_suffix"]
                ):
                    reasons.append("image_format_mismatch")
                if (
                    image_result["width"] is not None
                    and image_result["height"] is not None
                    and not image_result["expected_resolution_ok"]
                ):
                    reasons.append("unexpected_resolution")
                if (
                    not image_result["strict_decode_ok"]
                    and (
                        image_result["image_format"] is None
                        or image_result["width"] is None
                        or image_result["height"] is None
                    )
                ):
                    reasons.append("image_header_unreadable")
                if suffix in {".jpg", ".jpeg"} and not image_result["jpeg_soi_ok"]:
                    reasons.append("jpeg_soi_missing")
                if suffix in {".jpg", ".jpeg"} and not image_result["jpeg_eoi_ok"]:
                    reasons.append("jpeg_eoi_missing")

                declared_match = (
                    label_result["declared_width"] == image_result["width"]
                    and label_result["declared_height"] == image_result["height"]
                )
                if label_result["json_parse_ok"] and not declared_match:
                    reasons.append("declared_image_size_mismatch")

                reasons = sorted(set(reasons))
                quarantine_reason = ";".join(reasons)
                trainable = not reasons
                detector_trainable = (
                    trainable and label_result["valid_mokja_count"] > 0
                )
                surface_aux_trainable = (
                    trainable and label_result["valid_surface_count"] > 0
                )
                bbox_rows = label_result["bbox_rows"]

                manifest_row = {
                    "archive_key": key,
                    "stem": stem,
                    "type": label_result["type"],
                    "clslevel": label_result["clslevel"],
                    "waterlevel_m": label_result["waterlevel_m"],
                    "source_zip_uri": source_input["uri"],
                    "source_generation": source_input["generation"],
                    "source_member": source_entry["name"],
                    "source_member_crc32": source_entry["member_crc32"],
                    "source_member_bytes": len(image_raw),
                    "label_zip_uri": label_input["uri"],
                    "label_generation": label_input["generation"],
                    "label_member": label_entry["name"],
                    "label_member_crc32": label_entry["member_crc32"],
                    "label_member_bytes": len(label_raw),
                    "label_sha256": label_sha256,
                    "image_name": source_entry["name"],
                    "image_suffix": suffix,
                    "image_format": image_result["image_format"],
                    "width": image_result["width"],
                    "height": image_result["height"],
                    "image_sha256": image_result["image_sha256"],
                    "image_uuid": label_result["image_uuid"],
                    "image_id": label_result["image_id"],
                    "frame_id": label_result["frame_id"],
                    "json_parse_ok": label_result["json_parse_ok"],
                    "schema_ok": label_result["schema_ok"],
                    "strict_decode_ok": image_result["strict_decode_ok"],
                    "trainable": trainable,
                    "detector_trainable": detector_trainable,
                    "surface_aux_trainable": surface_aux_trainable,
                    "quarantine_reason": quarantine_reason,
                    "obstruction": label_result["obstruction"],
                    "annotation_count": label_result["annotation_count"],
                    "mokja_count": label_result["mokja_count"],
                    "surface_count": label_result["surface_count"],
                    "valid_mokja_count": label_result["valid_mokja_count"],
                    "valid_surface_count": label_result["valid_surface_count"],
                    "bbox_out_of_range": any(r["out_of_range"] for r in bbox_rows),
                    "bbox_degenerate": any(r["degenerate"] for r in bbox_rows),
                    "bbox_sanitized": any(r["sanitized"] for r in bbox_rows),
                    "bbox_issue_count": sum(bool(r["issue"]) for r in bbox_rows),
                }
                manifest_writer.append(manifest_row)

                integrity_writer.append({
                    "archive_key": key,
                    "stem": stem,
                    "source_zip_uri": source_input["uri"],
                    "source_member": source_entry["name"],
                    "image_name": source_entry["name"],
                    "image_suffix": suffix,
                    "source_member_bytes": len(image_raw),
                    "source_member_crc32": source_entry["member_crc32"],
                    "image_sha256": image_result["image_sha256"],
                    "image_format": image_result["image_format"],
                    "image_mode": image_result["image_mode"],
                    "width": image_result["width"],
                    "height": image_result["height"],
                    "strict_decode_ok": image_result["strict_decode_ok"],
                    "strict_error_type": image_result["strict_error_type"],
                    "strict_error_message": image_result["strict_error_message"],
                    "jpeg_soi_ok": image_result["jpeg_soi_ok"],
                    "jpeg_eoi_ok": image_result["jpeg_eoi_ok"],
                    "format_matches_suffix": image_result["format_matches_suffix"],
                    "expected_resolution_ok": image_result["expected_resolution_ok"],
                    "declared_width": label_result["declared_width"],
                    "declared_height": label_result["declared_height"],
                    "declared_size_matches": declared_match,
                    "trainable": trainable,
                    "quarantine_reason": quarantine_reason,
                })

                counters["json_parse_failure_count"] += int(
                    not label_result["json_parse_ok"]
                )
                counters["json_schema_failure_count"] += int(
                    not label_result["schema_ok"]
                )
                counters["strict_decode_failure_count"] += int(
                    not image_result["strict_decode_ok"]
                )
                counters["unexpected_resolution_count"] += int(
                    image_result["width"] is not None
                    and image_result["height"] is not None
                    and not image_result["expected_resolution_ok"]
                )
                counters["image_format_failure_count"] += int(
                    image_result["image_format"] is not None
                    and not image_result["format_matches_suffix"]
                )

                if reasons:
                    quarantine_items.append({
                        "archive_key": key,
                        "stem": stem,
                        "source_zip_uri": source_input["uri"],
                        "source_member": source_entry["name"],
                        "image_name": source_entry["name"],
                        "image_bytes": len(image_raw),
                        "image_sha256": image_result["image_sha256"],
                        "reason_code": reasons[0],
                        "reason_codes": reasons,
                        "error_type": image_result["strict_error_type"],
                        "error_message": image_result["strict_error_message"],
                        "json_schema_errors": label_result["schema_errors"],
                        "strict_decode_ok": image_result["strict_decode_ok"],
                        "trainable": False,
                        "detected_at_utc": iso_utc(),
                        "manual_review_status": "pending",
                        "source_modified": False,
                    })

                if index % PROGRESS_EVERY == 0 or index == expected_pairs:
                    elapsed = time.perf_counter() - started_records
                    rate = index / max(elapsed, 1e-9)
                    print(
                        f"[{key}] {index:,}/{expected_pairs:,} "
                        f"({rate:.1f} pair/s), quarantine={len(quarantine_items):,}"
                    )
                    refresh_run_lease()
                del image_raw, label_raw

        manifest_writer.close()
        integrity_writer.close()
        bbox_writer.close()
        open_writers.clear()

        if manifest_writer.row_count != expected_pairs:
            raise AuditError(
                f"{key} manifest rows 불일치: "
                f"{manifest_writer.row_count:,} != {expected_pairs:,}"
            )
        if integrity_writer.row_count != expected_pairs:
            raise AuditError(f"{key} image integrity rows 불일치")

        current_source = input_descriptor_for_key(key, "source")
        current_label = input_descriptor_for_key(key, "label")
        original_unchanged = all(
            current[field] == captured[field]
            for current, captured in [
                (current_source, source_input),
                (current_label, label_input),
            ]
            for field in ["object", "generation", "size", "crc32c"]
        )
        if not original_unchanged:
            raise IOError(f"{key} 처리 중 입력 GCS 객체가 변경되었습니다.")

        gate_pass = (
            counters["json_parse_failure_count"] == 0
            and counters["json_schema_failure_count"] == 0
            and counters["unexpected_resolution_count"] == 0
            and counters["image_format_failure_count"] == 0
            and counters["bbox_parse_error_count"] == 0
            and original_unchanged
        )
        status = (
            "PASS_WITH_QUARANTINE"
            if gate_pass and quarantine_items
            else "PASS"
            if gate_pass
            else "AUDIT_COMPLETE_WITH_GATE_ERRORS"
        )
        finished_at = iso_utc()
        audit_row = {
            "run_id": RUN_ID,
            "archive_key": key,
            "status": status,
            "gate_pass": gate_pass,
            "started_at_utc": started_at,
            "finished_at_utc": finished_at,
            "duration_seconds": time.perf_counter() - started_clock,
            "source_zip_uri": source_input["uri"],
            "label_zip_uri": label_input["uri"],
            "source_expected_bytes": expected_source,
            "source_actual_bytes": source_download["size"],
            "label_expected_bytes": expected_label,
            "label_actual_bytes": label_download["size"],
            "source_generation": source_input["generation"],
            "label_generation": label_input["generation"],
            "source_gcs_crc32c": source_input["crc32c"],
            "label_gcs_crc32c": label_input["crc32c"],
            "source_zip_sha256": source_download["sha256"],
            "label_zip_sha256": label_download["sha256"],
            "source_member_count": len(source_map),
            "label_member_count": len(label_map),
            "pair_count": expected_pairs,
            "only_image_count": 0,
            "only_json_count": 0,
            "duplicate_image_stem_count": 0,
            "duplicate_json_stem_count": 0,
            "zip_security_ok": True,
            "source_member_crc_verified": True,
            "label_member_crc_verified": True,
            "json_parse_failure_count": counters["json_parse_failure_count"],
            "json_schema_failure_count": counters["json_schema_failure_count"],
            "strict_decode_failure_count": counters["strict_decode_failure_count"],
            "unexpected_resolution_count": counters["unexpected_resolution_count"],
            "image_format_failure_count": counters["image_format_failure_count"],
            "quarantine_count": len(quarantine_items),
            "annotation_count": counters["annotation_count"],
            "bbox_parse_error_count": counters["bbox_parse_error_count"],
            "bbox_out_of_range_count": counters["bbox_out_of_range_count"],
            "bbox_crosses_frame_count": counters["bbox_crosses_frame_count"],
            "bbox_degenerate_count": counters["bbox_degenerate_count"],
            "bbox_sanitized_count": counters["bbox_sanitized_count"],
            "original_inputs_unchanged": original_unchanged,
        }
        write_single_row_parquet(
            archive_audit_local, ARCHIVE_AUDIT_SCHEMA, audit_row
        )
        quarantine_payload = {
            "schema_version": SCHEMA_VERSION,
            "run_id": RUN_ID,
            "archive_key": key,
            "original_inputs_modified": False,
            "quarantined_pair_count": len(quarantine_items),
            "items": sorted(quarantine_items, key=lambda item: item["stem"]),
        }
        write_json_file(quarantine_local, quarantine_payload)

        artifact_specs = {
            "raw_manifest": (
                manifest_local, "raw_manifest", manifest_writer.row_count
            ),
            "image_integrity": (
                image_integrity_local,
                "image_integrity",
                integrity_writer.row_count,
            ),
            "bbox_quality": (
                bbox_local, "bbox_quality", bbox_writer.row_count
            ),
            "archive_audit": (
                archive_audit_local, "archive_audit", 1
            ),
            "quarantine": (
                quarantine_local, "quarantine", len(quarantine_items)
            ),
        }
        artifacts = {}
        for name, (local_path, artifact_type, row_count) in artifact_specs.items():
            object_name = f"{attempt_prefix}/{local_path.name}"
            artifacts[name] = upload_immutable(
                local_path, object_name, artifact_type, row_count
            )
            print(f"[{key}] uploaded {name}: {artifacts[name]['uri']}")
        refresh_run_lease()

        commit_payload = {
            "status": "AUDIT_COMPLETE",
            "gate_pass": gate_pass,
            "audit_status": status,
            "run_id": RUN_ID,
            "session_id": SESSION_ID,
            "attempt_id": attempt_id,
            "key": key,
            "config_sha256": CONFIG_SHA256,
            "source_inventory_sha256": SOURCE_INVENTORY_SHA256,
            "started_at_utc": started_at,
            "completed_at_utc": finished_at,
            "inputs": {
                "source": {**source_input, "sha256": source_download["sha256"]},
                "label": {**label_input, "sha256": label_download["sha256"]},
            },
            "counts": {
                "raw_rows": manifest_writer.row_count,
                "bbox_rows": bbox_writer.row_count,
                "quarantine_rows": len(quarantine_items),
                **dict(counters),
            },
            "artifacts": artifacts,
            "original_inputs_modified": False,
        }
        commit_name = commit_object_name(key)
        try:
            upload_json_create_only(commit_payload, commit_name, "key_commit")
        except IOError:
            winner = verify_commit(key)
            if winner is None:
                raise
            print(f"[{key}] 다른 세션의 유효 commit을 채택합니다.")
            commit_payload = winner

        verified = verify_commit(key)
        if verified is None:
            raise IOError(f"{key} commit 검증 실패")
        print(
            f"[{key}] COMMIT {verified['audit_status']} / "
            f"rows={verified['counts']['raw_rows']:,}"
        )
        if CLEAN_LOCAL_AFTER_SUCCESS:
            safe_remove_tree(stage)
        gc.collect()
        return verified
    except Exception as exc:
        for writer in open_writers:
            try:
                writer.close()
            except Exception:
                pass
        try:
            report = upload_failure_report(key, attempt_id, started_at, exc)
            print("Failure report:", report["uri"])
        except Exception as report_exc:
            print("WARNING: failure report 업로드 실패:", repr(report_exc))
        if CLEAN_LOCAL_AFTER_FAILURE and stage.exists():
            safe_remove_tree(stage)
        raise


In [ ]:
SOURCE_INVENTORY, SOURCE_INVENTORY_SHA256 = collect_source_inventory()
RUN_CONFIG = ensure_run_config(SOURCE_INVENTORY, SOURCE_INVENTORY_SHA256)

print("Source inventory: 22 source + 22 label objects")
print("Source bytes:", f"{sum(x['size'] for x in SOURCE_INVENTORY if x['kind']=='source'):,}")
print("Label bytes:", f"{sum(x['size'] for x in SOURCE_INVENTORY if x['kind']=='label'):,}")
print("Inventory SHA-256:", SOURCE_INVENTORY_SHA256)

completed = {}
pending = []
for key in EXPECTED_KEYS:
    commit = verify_commit(key)
    if commit is None:
        pending.append(key)
    else:
        completed[key] = commit
print("Verified completed keys:", len(completed), sorted(completed))
print("Pending keys:", len(pending), pending)


Source inventory: 22 source + 22 label objects
Source bytes: 126,148,468,213
Label bytes: 273,886,103
Inventory SHA-256: 8ee696e2de736dd88c53d50115a66d778a888ad1ab9315d731d2ee6bca741f39
Verified completed keys: 0 []
Pending keys: 22 ['A_0005', 'A_0510', 'A_1015', 'B_0003', 'B_0306', 'B_0609', 'C_0005', 'C_0510', 'C_1015', 'E_0003', 'E_0306', 'E_0610', 'F_0002', 'F_0204', 'G_0002', 'G_0204', 'H_0007', 'H_0714', 'H_1420', 'S_0007', 'S_0714', 'S_1420']


## 2. 22개 archive pair 전수 감사

아래 셀이 가장 오래 걸립니다. 진행 중 Colab 연결이 끊기면 같은 RUN_ID로
노트북을 위에서부터 다시 실행하십시오. 이미 commit된 key는 입력 generation과
모든 산출물 hash를 확인한 뒤 SKIP됩니다.

MAX_KEYS_THIS_SESSION에 정수를 넣으면 이번 세션에서 처리할 key 수를 제한할 수 있습니다.


In [ ]:
selected = list(KEYS_TO_RUN) if KEYS_TO_RUN else list(EXPECTED_KEYS)
unknown = sorted(set(selected) - set(EXPECTED_KEYS))
if unknown:
    raise ValueError(f"알 수 없는 KEYS_TO_RUN: {unknown}")

processed_now = 0
try:
    if CURRENT_LEASE_GENERATION is None:
        acquire_run_lease()
    for key in selected:
        if MAX_KEYS_THIS_SESSION is not None and processed_now >= MAX_KEYS_THIS_SESSION:
            break
        before = verify_commit(key)
        process_archive_pair(key)
        if before is None:
            processed_now += 1
    refresh_run_lease()

    verified_commits = {}
    for key in EXPECTED_KEYS:
        commit = verify_commit(key)
        if commit is not None:
            verified_commits[key] = commit
    pending_after = [key for key in EXPECTED_KEYS if key not in verified_commits]
    gate_failed_after = [
        key for key, commit in verified_commits.items() if not commit["gate_pass"]
    ]
    print("This session processed:", processed_now)
    print("Total verified commits:", len(verified_commits), "/ 22")
    print("Still pending:", pending_after)
    print("Committed keys with gate errors:", gate_failed_after)
    if not pending_after and not gate_failed_after:
        print("All archive-pair audits committed. Final merge 셀로 진행하세요.")
    else:
        print("최종화 준비가 끝나지 않았습니다. 다음 세션에서 같은 RUN_ID로 재개하세요.")
finally:
    release_run_lease()


Run lease acquired: gs://YOUR_GCS_BUCKET/dcc/runs/20260825-manifest-v1/lease.json session 7e83a098

[A_0005] download source (6.232 GiB)
[A_0005] download labels (0.020 GiB)
[A_0005] inspect ZIP directory/security; record read verifies every member CRC
[A_0005] 1,000/24,068 (93.5 pair/s), quarantine=0
[A_0005] 2,000/24,068 (89.0 pair/s), quarantine=0
[A_0005] 3,000/24,068 (89.4 pair/s), quarantine=0
[A_0005] 4,000/24,068 (88.8 pair/s), quarantine=0
[A_0005] 5,000/24,068 (88.5 pair/s), quarantine=0
[A_0005] 6,000/24,068 (88.6 pair/s), quarantine=0
[A_0005] 7,000/24,068 (88.1 pair/s), quarantine=0
[A_0005] 8,000/24,068 (88.1 pair/s), quarantine=1
[A_0005] 9,000/24,068 (87.8 pair/s), quarantine=2
[A_0005] 10,000/24,068 (87.4 pair/s), quarantine=2
[A_0005] 11,000/24,068 (87.3 pair/s), quarantine=2
[A_0005] 12,000/24,068 (87.5 pair/s), quarantine=2
[A_0005] 13,000/24,068 (87.6 pair/s), quarantine=2
[A_0005] 14,000/24,068 (87.9 pair/s), quarantine=2
[A_0005] 15,000/24,068 (87.8 pair/s), quar

## 3. 최종 병합·Global Gate·Seal

22개 commit이 모두 있고 각 commit의 gate_pass가 True일 때 실행됩니다.
key별 Parquet을 고정 schema 순서로 병합하고, 308,085행·중복·수위·bbox·격리
회귀 수치를 다시 계산합니다. 6개 canonical 산출물을 검증한 뒤 seal을 마지막에 게시합니다.


In [ ]:
def download_artifact_verified(descriptor, destination):
    verify_remote_artifact(descriptor)
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    part = destination.with_name(destination.name + ".part")
    if part.exists():
        part.unlink()
    blob = BUCKET.blob(descriptor["object"])
    blob.download_to_filename(
        str(part),
        if_generation_match=int(descriptor["generation"]),
        checksum="crc32c",
    )
    hashes = local_file_hashes(part)
    if (
        hashes["size"] != descriptor["size"]
        or hashes["sha256"] != descriptor["sha256"]
        or hashes["crc32c"] != descriptor["crc32c"]
    ):
        raise IOError(f"산출물 다운로드 검증 실패: {descriptor['uri']}")
    os.replace(part, destination)
    return destination

def concatenate_parquet_files(inputs, output, schema):
    output = Path(output)
    writer = pq.ParquetWriter(
        output,
        schema,
        compression="zstd",
        use_dictionary=True,
        write_statistics=True,
    )
    rows = 0
    try:
        for input_path in inputs:
            parquet_file = pq.ParquetFile(input_path)
            if not parquet_file.schema_arrow.equals(schema, check_metadata=False):
                raise TypeError(f"Parquet schema 불일치: {input_path}")
            for batch in parquet_file.iter_batches(batch_size=65_536):
                table = pa.Table.from_batches([batch], schema=schema)
                writer.write_table(table)
                rows += batch.num_rows
    finally:
        writer.close()
    return rows

def validate_master_outputs(
    manifest_path,
    integrity_path,
    bbox_path,
    audit_path,
    quarantine_payload,
):
    summary = Counter()
    stems = set()
    integrity_stems = set()
    strict_failure_stems = set()
    not_trainable_stems = set()
    type_values = set()
    suffix_counts = Counter()
    mokja_distribution = Counter()
    surface_distribution = Counter()
    waterlevels = set()
    bbox_file_flags = set()
    pair_index = hashlib.sha256()
    known_row = None
    input_by_key_kind = {
        (item["key"], item["kind"]): item for item in SOURCE_INVENTORY
    }

    columns = [
        "archive_key", "stem", "type", "clslevel", "waterlevel_m",
        "source_zip_uri", "source_member", "label_zip_uri", "label_member",
        "source_generation", "label_generation", "source_member_crc32",
        "label_member_crc32", "source_member_bytes", "label_member_bytes",
        "image_suffix", "width", "height", "image_sha256", "label_sha256",
        "json_parse_ok", "schema_ok", "strict_decode_ok", "trainable",
        "detector_trainable", "surface_aux_trainable",
        "quarantine_reason", "obstruction", "annotation_count",
        "mokja_count", "surface_count", "valid_mokja_count",
        "valid_surface_count",
    ]
    manifest_file = pq.ParquetFile(manifest_path)
    for batch in manifest_file.iter_batches(batch_size=32_768, columns=columns):
        rows = pa.Table.from_batches([batch]).to_pylist()
        for row in rows:
            summary["raw_rows"] += 1
            stem = row["stem"]
            if stem in stems:
                summary["duplicate_stems"] += 1
            stems.add(stem)
            type_values.add(row["type"])
            suffix_counts[row["image_suffix"]] += 1
            mokja_distribution[row["mokja_count"]] += 1
            surface_distribution[row["surface_count"]] += 1
            summary["annotation_count"] += row["annotation_count"]
            summary["json_parse_failures"] += int(not row["json_parse_ok"])
            summary["json_schema_failures"] += int(not row["schema_ok"])
            if not all([
                row["source_zip_uri"],
                row["source_member"],
                row["label_zip_uri"],
                row["label_member"],
                row["source_member_crc32"],
                row["label_member_crc32"],
                row["image_sha256"],
                row["label_sha256"],
            ]) or row["source_member_bytes"] <= 0 or row["label_member_bytes"] <= 0:
                summary["reconstruction_field_errors"] += 1
            expected_source = input_by_key_kind[(row["archive_key"], "source")]
            expected_label = input_by_key_kind[(row["archive_key"], "label")]
            if (
                row["source_generation"] != expected_source["generation"]
                or row["label_generation"] != expected_label["generation"]
            ):
                summary["generation_lineage_errors"] += 1
            if row["width"] is not None and row["height"] is not None:
                summary["unexpected_resolution"] += int(
                    row["width"] != 1920 or row["height"] != 1080
                )
            elif row["strict_decode_ok"]:
                summary["unexpected_resolution"] += 1
            if row["waterlevel_m"] is None or not math.isfinite(row["waterlevel_m"]):
                summary["invalid_waterlevel"] += 1
            else:
                waterlevels.add(float(row["waterlevel_m"]))
            if not row["strict_decode_ok"]:
                strict_failure_stems.add(stem)
                summary["strict_decode_failures"] += 1
                if row["trainable"] or not row["quarantine_reason"]:
                    summary["decode_policy_errors"] += 1
            if not row["trainable"]:
                not_trainable_stems.add(stem)
            if row["trainable"] and not (
                row["json_parse_ok"]
                and row["schema_ok"]
                and row["strict_decode_ok"]
            ):
                summary["trainable_policy_errors"] += 1
            expected_detector_trainable = (
                row["trainable"] and row["valid_mokja_count"] > 0
            )
            expected_surface_trainable = (
                row["trainable"] and row["valid_surface_count"] > 0
            )
            if row["detector_trainable"] != expected_detector_trainable:
                summary["detector_trainable_policy_errors"] += 1
            if row["surface_aux_trainable"] != expected_surface_trainable:
                summary["surface_trainable_policy_errors"] += 1
            obstruction = row["obstruction"] or []
            summary["obstruction_tags"] += len(obstruction)
            summary["multi_obstruction_rows"] += int(len(obstruction) > 1)
            summary["empty_obstruction_tags"] += sum(tag == "" for tag in obstruction)
            real_tags = [tag for tag in obstruction if tag not in {"", "해당없음"}]
            summary["obstruction_contradictions"] += int(
                "해당없음" in obstruction and bool(real_tags)
            )
            pair_index.update(
                (
                    f"{row['archive_key']}\t{stem}\t"
                    f"{row['image_sha256']}\t{row['label_sha256']}\n"
                ).encode("utf-8")
            )
            if stem == KNOWN_H_0007["stem"]:
                known_row = row

    integrity_file = pq.ParquetFile(integrity_path)
    for batch in integrity_file.iter_batches(
        batch_size=32_768,
        columns=["archive_key", "stem"],
    ):
        for row in pa.Table.from_batches([batch]).to_pylist():
            summary["image_integrity_rows"] += 1
            stem = row["stem"]
            if stem in integrity_stems:
                summary["image_integrity_duplicate_stems"] += 1
            integrity_stems.add(stem)

    bbox_name_counts = Counter()
    bbox_file = pq.ParquetFile(bbox_path)
    for batch in bbox_file.iter_batches(
        batch_size=65_536,
        columns=[
            "stem", "annotation_name", "parse_ok", "out_of_range",
            "crosses_frame", "degenerate", "sanitized", "usable_after_clip",
            "clipped_x1", "clipped_y1", "clipped_x2", "clipped_y2",
        ],
    ):
        for row in pa.Table.from_batches([batch]).to_pylist():
            summary["bbox_rows"] += 1
            summary["bbox_parse_errors"] += int(not row["parse_ok"])
            summary["bbox_out_of_range"] += int(row["out_of_range"])
            summary["bbox_crosses_frame"] += int(row["crosses_frame"])
            summary["bbox_degenerate"] += int(row["degenerate"])
            summary["bbox_sanitized"] += int(row["sanitized"])
            if row["parse_ok"]:
                clipped = [
                    row["clipped_x1"], row["clipped_y1"],
                    row["clipped_x2"], row["clipped_y2"],
                ]
                if not all(
                    value is not None and math.isfinite(value) and 0 <= value <= 1
                    for value in clipped
                ):
                    summary["bbox_clip_policy_errors"] += 1
                if row["sanitized"] != (
                    row["crosses_frame"] and not row["degenerate"]
                ):
                    summary["bbox_sanitize_policy_errors"] += 1
                if row["usable_after_clip"] != (not row["degenerate"]):
                    summary["bbox_usable_policy_errors"] += 1
            if row["out_of_range"]:
                bbox_file_flags.add(row["stem"])
                bbox_name_counts[row["annotation_name"]] += 1

    audit_table = pq.read_table(audit_path)
    audit_rows = audit_table.to_pylist()
    summary["archive_rows"] = len(audit_rows)
    summary["only_image_count"] = sum(r["only_image_count"] for r in audit_rows)
    summary["only_json_count"] = sum(r["only_json_count"] for r in audit_rows)
    summary["archive_gate_failures"] = sum(not r["gate_pass"] for r in audit_rows)
    summary["input_mutation_count"] = sum(
        not r["original_inputs_unchanged"] for r in audit_rows
    )

    quarantine_items = quarantine_payload["items"]
    quarantine_stems = {item["stem"] for item in quarantine_items}
    decode_quarantine_stems = {
        item["stem"]
        for item in quarantine_items
        if "strict_decode_failed" in item.get("reason_codes", [])
    }
    summary["quarantine_count"] = len(quarantine_items)
    summary["duplicate_quarantine_items"] = (
        len(quarantine_items) - len(quarantine_stems)
    )
    summary["unique_stems"] = len(stems)
    summary["bbox_out_of_range_files"] = len(bbox_file_flags)
    summary["waterlevel_unique"] = len(waterlevels)
    summary["waterlevel_min"] = min(waterlevels) if waterlevels else None
    summary["waterlevel_max"] = max(waterlevels) if waterlevels else None

    hard_assertions = {
        "raw_rows": summary["raw_rows"] == EXPECTED_RAW_ROWS,
        "unique_stems": len(stems) == EXPECTED_RAW_ROWS,
        "duplicate_stems_zero": summary["duplicate_stems"] == 0,
        "image_integrity_rows": (
            summary["image_integrity_rows"] == EXPECTED_RAW_ROWS
        ),
        "image_integrity_unique_stems": (
            len(integrity_stems) == EXPECTED_RAW_ROWS
        ),
        "image_integrity_duplicate_stems_zero": (
            summary["image_integrity_duplicate_stems"] == 0
        ),
        "image_integrity_manifest_stems_exact": integrity_stems == stems,
        "types": type_values == EXPECTED_TYPES,
        "only_image_zero": summary["only_image_count"] == 0,
        "only_json_zero": summary["only_json_count"] == 0,
        "json_parse_zero": summary["json_parse_failures"] == 0,
        "json_schema_zero": summary["json_schema_failures"] == 0,
        "bbox_parse_zero": summary["bbox_parse_errors"] == 0,
        "resolution": summary["unexpected_resolution"] == 0,
        "waterlevel_finite": summary["invalid_waterlevel"] == 0,
        "reconstruction_fields": summary["reconstruction_field_errors"] == 0,
        "generation_lineage": summary["generation_lineage_errors"] == 0,
        "archive_rows": summary["archive_rows"] == 22,
        "archive_gates": summary["archive_gate_failures"] == 0,
        "inputs_unchanged": summary["input_mutation_count"] == 0,
        "decode_policy": summary["decode_policy_errors"] == 0,
        "trainable_policy": summary["trainable_policy_errors"] == 0,
        "detector_trainable_policy": (
            summary["detector_trainable_policy_errors"] == 0
        ),
        "surface_trainable_policy": (
            summary["surface_trainable_policy_errors"] == 0
        ),
        "bbox_clip_policy": summary["bbox_clip_policy_errors"] == 0,
        "bbox_sanitize_policy": summary["bbox_sanitize_policy_errors"] == 0,
        "bbox_usable_policy": summary["bbox_usable_policy_errors"] == 0,
        "quarantine_unique": summary["duplicate_quarantine_items"] == 0,
        "not_trainable_quarantine_exact": not_trainable_stems == quarantine_stems,
        "decode_quarantine_exact": strict_failure_stems == decode_quarantine_stems,
        "decode_quarantine_subset": strict_failure_stems <= quarantine_stems,
    }

    if known_row is None:
        hard_assertions["known_h_present"] = False
    else:
        hard_assertions.update({
            "known_h_sha256": (
                known_row["image_sha256"] == KNOWN_H_0007["image_sha256"]
            ),
            "known_h_bytes": (
                known_row["source_member_bytes"] == KNOWN_H_0007["image_bytes"]
            ),
            "known_h_decode_failure": not known_row["strict_decode_ok"],
            "known_h_not_trainable": not known_row["trainable"],
            "known_h_quarantined": KNOWN_H_0007["stem"] in quarantine_stems,
        })

    regression_assertions = {
        "suffix_counts": dict(suffix_counts) == EXPECTED_SUFFIX_COUNTS,
        "annotation_count": summary["annotation_count"] == EXPECTED_ANNOTATIONS,
        "bbox_row_count": summary["bbox_rows"] == EXPECTED_ANNOTATIONS,
        "mokja_distribution": dict(mokja_distribution) == EXPECTED_MOKJA_COUNT_DISTRIBUTION,
        "surface_distribution": dict(surface_distribution) == EXPECTED_SURFACE_COUNT_DISTRIBUTION,
        "bbox_out_of_range": summary["bbox_out_of_range"] == EXPECTED_BBOX_OUT_OF_RANGE,
        "bbox_out_of_range_files": (
            summary["bbox_out_of_range_files"] == EXPECTED_BBOX_OUT_OF_RANGE_FILES
        ),
        "bbox_out_of_range_by_name": (
            dict(bbox_name_counts) == EXPECTED_BBOX_OUT_OF_RANGE_BY_NAME
        ),
        "obstruction_tags": summary["obstruction_tags"] == EXPECTED_OBSTRUCTION_TAGS,
        "multi_obstruction_rows": (
            summary["multi_obstruction_rows"] == EXPECTED_MULTI_OBSTRUCTION_ROWS
        ),
        "empty_obstruction_tags": (
            summary["empty_obstruction_tags"] == EXPECTED_EMPTY_OBSTRUCTION_TAGS
        ),
        "obstruction_contradictions": (
            summary["obstruction_contradictions"]
            == EXPECTED_OBSTRUCTION_CONTRADICTIONS
        ),
        "waterlevel_min": summary["waterlevel_min"] == EXPECTED_WATERLEVEL_MIN,
        "waterlevel_max": summary["waterlevel_max"] == EXPECTED_WATERLEVEL_MAX,
        "waterlevel_unique": summary["waterlevel_unique"] == EXPECTED_WATERLEVEL_UNIQUE,
    }
    if not all(hard_assertions.values()):
        failed = [key for key, value in hard_assertions.items() if not value]
        raise AssertionError(f"Global hard Gate 실패: {failed}")
    if STRICT_REGRESSION_CHECKS and not all(regression_assertions.values()):
        failed = [key for key, value in regression_assertions.items() if not value]
        raise AssertionError(
            "기존 전수 조사값 재현 실패: "
            f"{failed}. 정의와 원시 데이터를 대조한 뒤 새 OUTPUT_VERSION을 사용하세요."
        )
    return {
        "counts": dict(summary),
        "type_values": sorted(type_values),
        "suffix_counts": dict(suffix_counts),
        "mokja_count_distribution": dict(mokja_distribution),
        "surface_count_distribution": dict(surface_distribution),
        "bbox_out_of_range_by_name": dict(bbox_name_counts),
        "strict_decode_failure_stems": sorted(strict_failure_stems),
        "quarantine_stems": sorted(quarantine_stems),
        "pair_index_sha256": pair_index.hexdigest(),
        "hard_assertions": hard_assertions,
        "regression_assertions": regression_assertions,
    }

def finalize_manifest():
    if not STRICT_REGRESSION_CHECKS:
        raise RuntimeError(
            "canonical publish와 seal 생성에는 STRICT_REGRESSION_CHECKS=True가 필요합니다."
        )

    seal_object = f"dcc/audits/manifest_seal_{OUTPUT_VERSION}.json"
    current_inventory, current_inventory_sha = collect_source_inventory()
    if current_inventory_sha != SOURCE_INVENTORY_SHA256:
        raise IOError("최종화 시작 전 raw source inventory가 변경되었습니다.")

    commits = {}
    for key in EXPECTED_KEYS:
        commit = verify_commit(key)
        if commit is None:
            raise RuntimeError(f"미완료 key: {key}")
        if not commit["gate_pass"]:
            raise RuntimeError(f"archive gate 실패 commit: {key}")
        commits[key] = commit

    existing_seal = read_json_blob(seal_object)
    if existing_seal is not None:
        if (
            existing_seal.get("run_id") != RUN_ID
            or existing_seal.get("config_sha256") != CONFIG_SHA256
        ):
            raise RuntimeError(
                f"{gs_uri(seal_object)}가 다른 run/config으로 이미 존재합니다. "
                "OUTPUT_VERSION을 올리십시오."
            )
        existing_seal, _ = read_and_verify_immutable_json(
            seal_object, "manifest_seal"
        )
        if (
            existing_seal.get("source_inventory_sha256")
            != current_inventory_sha
            or existing_seal.get("source_inventory") != current_inventory
        ):
            raise IOError("기존 seal의 raw source inventory가 현재 입력과 다릅니다.")
        if existing_seal.get("completed_keys") != list(EXPECTED_KEYS):
            raise IOError("기존 seal의 completed_keys가 22개 기준과 다릅니다.")
        sealed_commits = existing_seal.get("key_commits", {})
        if set(sealed_commits) != set(EXPECTED_KEYS):
            raise IOError("기존 seal의 key commit 집합이 22개 기준과 다릅니다.")
        for key, commit in commits.items():
            descriptor = sealed_commits[key]
            verify_remote_artifact(descriptor)
            payload_sha256 = hashlib.sha256(
                canonical_json_bytes(commit)
            ).hexdigest()
            if descriptor.get("payload_sha256") != payload_sha256:
                raise IOError(f"기존 seal의 key commit payload hash 불일치: {key}")
        for descriptor in existing_seal["artifacts"].values():
            verify_remote_artifact(descriptor)
        validation = existing_seal.get("validation", {})
        hard_assertions = validation.get("hard_assertions", {})
        regression_assertions = validation.get("regression_assertions", {})
        if not hard_assertions or not all(hard_assertions.values()):
            raise IOError("기존 seal의 Global hard assertion이 완전한 PASS가 아닙니다.")
        if not regression_assertions or not all(regression_assertions.values()):
            raise IOError("기존 seal의 regression assertion이 완전한 PASS가 아닙니다.")
        if existing_seal.get("status") not in {"PASS", "PASS_WITH_QUARANTINE"}:
            raise IOError("기존 seal의 status가 최종 성공 상태가 아닙니다.")
        print("Existing manifest seal verified:", gs_uri(seal_object))
        return existing_seal

    final_root = LOCAL_RUN_ROOT / "_finalize"
    if final_root.exists():
        safe_remove_tree(final_root)
    final_root.mkdir(parents=True, exist_ok=False)
    shard_root = final_root / "shards"
    shard_root.mkdir()

    downloaded = {
        "raw_manifest": [],
        "image_integrity": [],
        "bbox_quality": [],
        "archive_audit": [],
    }
    shard_bytes_by_artifact = {
        name: sum(
            int(commits[key]["artifacts"][name]["size"])
            for key in EXPECTED_KEYS
        )
        for name in downloaded
    }
    quarantine_shard_bytes = sum(
        int(commits[key]["artifacts"]["quarantine"]["size"])
        for key in EXPECTED_KEYS
    )
    all_shard_bytes = sum(shard_bytes_by_artifact.values()) + quarantine_shard_bytes
    largest_merged_output = max(shard_bytes_by_artifact.values())
    required_free = all_shard_bytes + largest_merged_output + 5 * 1024**3
    free = shutil.disk_usage(final_root).free
    print(
        "Finalize disk preflight:",
        f"free={free/1024**3:.1f} GiB, required={required_free/1024**3:.1f} GiB",
    )
    if free < required_free:
        raise RuntimeError(
            "최종 병합 공간 부족: "
            f"free={free/1024**3:.1f} GiB, "
            f"required={required_free/1024**3:.1f} GiB"
        )
    quarantine_items = []
    for key in EXPECTED_KEYS:
        commit = commits[key]
        key_dir = shard_root / key
        key_dir.mkdir()
        for artifact_name in downloaded:
            desc = commit["artifacts"][artifact_name]
            local = key_dir / f"{artifact_name}.parquet"
            download_artifact_verified(desc, local)
            downloaded[artifact_name].append(local)
        q_path = key_dir / "quarantine.json"
        download_artifact_verified(commit["artifacts"]["quarantine"], q_path)
        q_payload = json.loads(q_path.read_text(encoding="utf-8"))
        quarantine_items.extend(q_payload["items"])
        q_path.unlink()
        refresh_run_lease()

    manifest_path = final_root / f"raw_manifest_{OUTPUT_VERSION}.parquet"
    integrity_path = final_root / f"image_integrity_{OUTPUT_VERSION}.parquet"
    bbox_path = final_root / f"bbox_quality_{OUTPUT_VERSION}.parquet"
    audit_path = final_root / f"archive_audit_{OUTPUT_VERSION}.parquet"
    quarantine_path = final_root / f"quarantine_{OUTPUT_VERSION}.json"
    summary_csv_path = final_root / f"archive_audit_summary_{OUTPUT_VERSION}.csv"

    def merge_and_remove_shards(name, output_path, schema):
        rows = concatenate_parquet_files(downloaded[name], output_path, schema)
        for shard_path in downloaded[name]:
            shard_path.unlink()
        downloaded[name].clear()
        gc.collect()
        refresh_run_lease()
        return rows

    final_rows = {}
    final_rows["raw_manifest"] = merge_and_remove_shards(
        "raw_manifest", manifest_path, MANIFEST_SCHEMA
    )
    final_rows["image_integrity"] = merge_and_remove_shards(
        "image_integrity", integrity_path, IMAGE_INTEGRITY_SCHEMA
    )
    final_rows["bbox_quality"] = merge_and_remove_shards(
        "bbox_quality", bbox_path, BBOX_SCHEMA
    )
    final_rows["archive_audit"] = merge_and_remove_shards(
        "archive_audit", audit_path, ARCHIVE_AUDIT_SCHEMA
    )
    quarantine_payload = {
        "schema_version": SCHEMA_VERSION,
        "run_id": RUN_ID,
        "original_inputs_modified": False,
        "quarantined_pair_count": len(quarantine_items),
        "items": sorted(
            quarantine_items,
            key=lambda item: (item["archive_key"], item["stem"]),
        ),
    }
    write_json_file(quarantine_path, quarantine_payload)

    audit_rows = pq.read_table(audit_path).to_pylist()
    with summary_csv_path.open("w", newline="", encoding="utf-8-sig") as writer_file:
        fields = [
            "archive_key", "status", "pair_count",
            "strict_decode_failure_count", "quarantine_count",
            "annotation_count", "bbox_out_of_range_count",
            "bbox_degenerate_count", "duration_seconds",
        ]
        csv_writer = csv.DictWriter(writer_file, fieldnames=fields)
        csv_writer.writeheader()
        for row in audit_rows:
            csv_writer.writerow({field: row[field] for field in fields})

    validation = validate_master_outputs(
        manifest_path,
        integrity_path,
        bbox_path,
        audit_path,
        quarantine_payload,
    )
    current_inventory, current_inventory_sha = collect_source_inventory()
    if current_inventory_sha != SOURCE_INVENTORY_SHA256:
        raise IOError("최종화 직전 raw source inventory가 변경되었습니다.")

    canonical_specs = {
        "raw_manifest": (
            manifest_path,
            f"dcc/manifests/raw_manifest_{OUTPUT_VERSION}.parquet",
            final_rows["raw_manifest"],
        ),
        "archive_audit": (
            audit_path,
            f"dcc/audits/archive_audit_{OUTPUT_VERSION}.parquet",
            final_rows["archive_audit"],
        ),
        "image_integrity": (
            integrity_path,
            f"dcc/audits/image_integrity_{OUTPUT_VERSION}.parquet",
            final_rows["image_integrity"],
        ),
        "bbox_quality": (
            bbox_path,
            f"dcc/audits/bbox_quality_{OUTPUT_VERSION}.parquet",
            final_rows["bbox_quality"],
        ),
        "quarantine": (
            quarantine_path,
            f"dcc/audits/quarantine_{OUTPUT_VERSION}.json",
            len(quarantine_items),
        ),
        "archive_summary_csv": (
            summary_csv_path,
            f"dcc/audits/archive_audit_summary_{OUTPUT_VERSION}.csv",
            22,
        ),
    }
    final_artifacts = {}
    for name, (local, object_name, rows) in canonical_specs.items():
        final_artifacts[name] = upload_immutable(
            local, object_name, name, rows
        )
        print("Published:", final_artifacts[name]["uri"])
        refresh_run_lease()

    status = (
        "PASS_WITH_QUARANTINE"
        if validation["counts"]["quarantine_count"]
        else "PASS"
    )
    key_commit_descriptors = {}
    for key, commit in commits.items():
        descriptor = immutable_blob_descriptor(
            BUCKET.blob(commit_object_name(key))
        )
        descriptor["payload_sha256"] = hashlib.sha256(
            canonical_json_bytes(commit)
        ).hexdigest()
        key_commit_descriptors[key] = descriptor
    seal_payload = {
        "status": status,
        "schema_version": SCHEMA_VERSION,
        "pipeline_version": PIPELINE_VERSION,
        "pipeline_build_sha256": PIPELINE_BUILD_SHA256,
        "notebook_name": NOTEBOOK_NAME,
        "run_id": RUN_ID,
        "output_version": OUTPUT_VERSION,
        "project_id": PROJECT_ID,
        "bucket_name": BUCKET_NAME,
        "config_sha256": CONFIG_SHA256,
        "source_inventory_sha256": SOURCE_INVENTORY_SHA256,
        "source_inventory": current_inventory,
        "completed_keys": list(EXPECTED_KEYS),
        "key_commits": key_commit_descriptors,
        "artifacts": final_artifacts,
        "validation": validation,
        "original_inputs_modified": False,
        "created_at_utc": iso_utc(),
    }
    seal_descriptor = upload_json_create_only(
        seal_payload, seal_object, "manifest_seal"
    )
    verified_seal, _ = read_and_verify_immutable_json(
        seal_object, "manifest_seal"
    )
    if verified_seal.get("status") != status:
        raise IOError("manifest seal 재검증 실패")
    print("Manifest seal:", seal_descriptor["uri"])
    print("Full Training content audit:", status)
    print("Raw rows:", f"{validation['counts']['raw_rows']:,}")
    print("Unique stems:", f"{validation['counts']['unique_stems']:,}")
    print("Quarantine:", f"{validation['counts']['quarantine_count']:,}")
    if CLEAN_LOCAL_AFTER_SUCCESS:
        safe_remove_tree(final_root)
    return verified_seal


In [ ]:
try:
    if CURRENT_LEASE_GENERATION is None:
        acquire_run_lease()
    FINAL_SEAL = finalize_manifest()
finally:
    release_run_lease()

print(json.dumps({
    "status": FINAL_SEAL["status"],
    "run_id": FINAL_SEAL["run_id"],
    "output_version": FINAL_SEAL["output_version"],
    "raw_rows": FINAL_SEAL["validation"]["counts"]["raw_rows"],
    "unique_stems": FINAL_SEAL["validation"]["counts"]["unique_stems"],
    "quarantine_count": FINAL_SEAL["validation"]["counts"]["quarantine_count"],
    "pair_index_sha256": FINAL_SEAL["validation"]["pair_index_sha256"],
}, ensure_ascii=False, indent=2))


Run lease acquired: gs://YOUR_GCS_BUCKET/dcc/runs/20260825-manifest-v1/lease.json session 7e83a098
Run lease released.


RuntimeError: archive gate 실패 commit: C_0510

## 완료 후 다음 단계

출력의 Full Training content audit가 PASS 또는 PASS_WITH_QUARANTINE이고,
raw_rows와 unique_stems가 모두 308,085인지 확인합니다.

생성되는 핵심 경로:

~~~text
gs://버킷/dcc/manifests/raw_manifest_v1.parquet
gs://버킷/dcc/audits/archive_audit_v1.parquet
gs://버킷/dcc/audits/image_integrity_v1.parquet
gs://버킷/dcc/audits/bbox_quality_v1.parquet
gs://버킷/dcc/audits/quarantine_v1.json
gs://버킷/dcc/audits/manifest_seal_v1.json
~~~

seal을 보존한 뒤 Manifest 기반 EDA와 A/B/C/E/F/G/H/S별 3장씩
총 24장 bbox 검수로 진행합니다. split과 모델 학습은 이 Gate가 통과한 뒤 시작합니다.


In [ ]:
c = verify_commit("C_0510")

gate_fields = [
    "json_parse_failure_count",
    "json_schema_failure_count",
    "unexpected_resolution_count",
    "image_format_failure_count",
    "bbox_parse_error_count",
    "quarantine_rows",
]

print("audit_status:", c["audit_status"])
print("gate_pass:", c["gate_pass"])
print({k: c["counts"].get(k, 0) for k in gate_fields})
print("original_inputs_modified:", c["original_inputs_modified"])

d = c["artifacts"]["quarantine"]
raw = BUCKET.blob(
    d["object"], generation=int(d["generation"])
).download_as_bytes(if_generation_match=int(d["generation"]))

q = json.loads(raw.decode("utf-8"))
print("quarantine reasons:", Counter(
    reason
    for item in q["items"]
    for reason in item.get("reason_codes", [])
))

audit_status: AUDIT_COMPLETE_WITH_GATE_ERRORS
gate_pass: False
{'json_parse_failure_count': 0, 'json_schema_failure_count': 0, 'unexpected_resolution_count': 0, 'image_format_failure_count': 153, 'bbox_parse_error_count': 0, 'quarantine_rows': 153}
original_inputs_modified: False
quarantine reasons: Counter({'image_format_mismatch': 153, 'jpeg_eoi_missing': 153, 'jpeg_soi_missing': 153})
